# hs_rag_qwen

Complete revised hybrid RAG notebook using BGE-M3 for retrieval and Qwen2.5-3B-Instruct for answer generation.

In [1]:
import pandas as pd
import math
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import torch
import gradio as gr
import faiss
from rank_bm25 import BM25Okapi
import re
import warnings
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from pathlib import Path
import zipfile
import json

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

warnings.filterwarnings('ignore')

# Create common project folders used throughout the notebook.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
SEARCH_REVIEW_DIR = Path("search_review")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SEARCH_REVIEW_DIR.mkdir(exist_ok=True)

# NLTK stop words are helpful for BM25 and word-overlap scoring.
# This keeps the notebook runnable even on a fresh machine.
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()


In [2]:
DEPARTMENT_NAME = "Computer Science and Electrical Engineering"
SOURCE_ROOT = Path("DepCourseInfo")
ZIP_PATH_CANDIDATES = [Path("DepCourseInfo.zip"), Path("/mnt/data/DepCourseInfo.zip")]
PDF_FOLDER_NAME = "downloaded_forms_and_worksheets"

# Prefixes that are common in CS, CE, cybersecurity, networking, data science, AI, and EE/engineering course material.
# Keep this broad because the EECS pages may contain interdisciplinary course documents too.
EECS_PREFIXES = {
    "CAP", "CDA", "CEN", "CGS", "CIS", "CNT", "COP", "COT", "CSC",
    "EEE", "EEL", "EGN", "EGS", "EIN", "EML", "IDC", "ISC", "STA", "MAP", "MAD"
}

# # Avoid carrying over biomedical/healthcare-oriented documents from the old topic.
# # A CS/EE course with a bioinformatics title is still kept if its prefix is part of EECS_PREFIXES.
# BIOMEDICAL_EXCLUDE_TERMS = {
#     "bme", "biomedical", "bioengineering", "bio-systems", "stem cell", "tissue engineering"
# }

def normalize_whitespace(text):
    return " ".join(str(text).replace("\x00", " ").split())


def clean_title_from_filename(filename):
    stem = Path(filename).stem
    stem = re.sub(r"[_-]+", " ", stem)
    stem = re.sub(r"\s+", " ", stem).strip()
    return stem.title()


def ensure_source_files_available(source_root=SOURCE_ROOT):
    """Use an existing DepCourseInfo folder, or unzip DepCourseInfo.zip if needed."""
    if source_root.exists():
        print(f"Using existing source folder: {source_root.resolve()}")
        return source_root

    zip_path = next((p for p in ZIP_PATH_CANDIDATES if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            "Could not find DepCourseInfo folder or DepCourseInfo.zip. "
            "Place the ZIP next to this notebook or update ZIP_PATH_CANDIDATES."
        )

    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(Path("."))

    if not source_root.exists():
        raise FileNotFoundError(f"Expected extracted folder was not found: {source_root}")

    print(f"Extracted source folder: {source_root.resolve()}")
    return source_root


def guess_document_type(filename, linked_url="", title=""):
    text = f"{filename} {linked_url} {title}".lower()

    if any(word in text for word in ["worksheet", "application", "audit", "form"]):
        return "program_form_or_worksheet"
    if "certificate" in text or "minor" in text or "concentration" in text:
        return "certificate_minor_or_concentration"
    if re.search(r"\b[a-z]{3}\s*[-_ ]?\d{4}", text):
        return "course_syllabus"
    if "syllab" in text:
        return "course_syllabus"
    return "department_document"


def is_relevant_eecs_document(prefix="", title="", filename="", doc_type=""):
    """Return True for CS/EE department material and False for biomedical/healthcare carryover."""
    prefix = str(prefix).upper().strip()
    combined = f"{title} {filename}".lower()

    if prefix:
        return prefix in EECS_PREFIXES

    # if any(term in combined for term in BIOMEDICAL_EXCLUDE_TERMS):
    #     return False

    # Keep program forms, worksheets, certificates, and general department files that do not have a course prefix.
    return True


def extract_pdf_text(pdf_path, max_pages=None):
    """Extract text from a PDF. Returns an empty string for scanned/unreadable PDFs."""
    if PdfReader is None:
        raise ImportError("pypdf is required. Install it with: pip install pypdf")

    try:
        reader = PdfReader(str(pdf_path))
        pages = reader.pages if max_pages is None else reader.pages[:max_pages]
        page_text = []
        for page in pages:
            try:
                page_text.append(page.extract_text() or "")
            except Exception:
                page_text.append("")
        return normalize_whitespace("\n".join(page_text))
    except Exception as exc:
        print(f"Warning: could not read PDF {pdf_path.name}: {exc}")
        return ""


def build_department_corpus(source_root=SOURCE_ROOT, data_dir=DATA_DIR):
    """
    Build a retrieval corpus for CS/EE department information.

    Output columns intentionally include the original template's required columns:
    - doc_id
    - text

    Additional metadata columns are kept for better summaries and inspection.
    """
    source_root = ensure_source_files_available(source_root)
    data_dir.mkdir(exist_ok=True)

    course_csv = source_root / "course_listings.csv"
    pdf_dir = source_root / PDF_FOLDER_NAME

    rows = []
    linked_file_lookup = {}

    if course_csv.exists():
        courses_df = pd.read_csv(course_csv)
        courses_df = courses_df.fillna("")

        # Map downloaded PDF filenames back to their source URLs when possible.
        for _, row in courses_df.iterrows():
            linked = str(row.get("linked_file_or_page", "")).strip()
            if linked.lower().endswith(".pdf"):
                linked_file_lookup[Path(linked).name.lower()] = linked

        # Add one clean course-listing document per unique course/title/source combination.
        course_docs = courses_df.copy()
        course_docs["course_key"] = (
            course_docs["course_code"].astype(str) + " | " +
            course_docs["title_or_context"].astype(str) + " | " +
            course_docs["level"].astype(str)
        )
        course_docs = course_docs.drop_duplicates("course_key")

        for i, row in course_docs.iterrows():
            prefix = str(row.get("prefix", "")).upper().strip()
            course_code = str(row.get("course_code", "")).strip()
            title = str(row.get("title_or_context", "")).strip()

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename="", doc_type="course_listing"):
                continue
            level = str(row.get("level", "")).strip()
            source_page = str(row.get("source_page", "")).strip()
            linked = str(row.get("linked_file_or_page", "")).strip()
            raw_text = str(row.get("raw_text", "")).strip()

            # Keep broad EECS results, but avoid empty/noisy rows.
            if not course_code and not title and not raw_text:
                continue

            doc_id = f"course_{len(rows)+1:04d}_{course_code.replace(' ', '_') or 'unknown'}"
            labeled_text = normalize_whitespace(f"""
Document_Type = course_listing,
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {row.get('number', '')},
Level = {level},
Source_Page = {source_page},
Linked_File_or_Page = {linked},
File_Name = ,
Content = {raw_text if raw_text else f'{course_code} {title}'}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": "course_listing",
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": str(row.get("number", "")).strip(),
                "level": level,
                "source_page": source_page,
                "linked_file_or_page": linked,
                "file_name": "",
                "text": labeled_text,
            })
    else:
        print(f"Warning: course listings file was not found: {course_csv}")

    if pdf_dir.exists():
        pdf_paths = sorted(pdf_dir.rglob("*.pdf"))
        print(f"Reading {len(pdf_paths)} PDFs from {pdf_dir} ...")

        for pdf_path in pdf_paths:
            file_name = pdf_path.name
            linked = linked_file_lookup.get(file_name.lower(), "")
            title = clean_title_from_filename(file_name)
            doc_type = guess_document_type(file_name, linked, title)
            pdf_text = extract_pdf_text(pdf_path)

            # Try to identify course code metadata from the filename or extracted text.
            course_match = re.search(r"\b([A-Z]{3})\s*[-_ ]?(\d{4})\b", f"{file_name} {pdf_text[:500]}", flags=re.IGNORECASE)
            prefix = course_match.group(1).upper() if course_match else ""
            number = course_match.group(2) if course_match else ""
            course_code = f"{prefix} {number}".strip()
            level = "graduate" if number.startswith(("5", "6")) else "undergraduate" if number.startswith(("1", "2", "3", "4")) else ""

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename=file_name, doc_type=doc_type):
                continue

            content = pdf_text if pdf_text else "No extractable PDF text was found. Use the filename and metadata for retrieval."
            doc_id = f"pdf_{len(rows)+1:04d}_{Path(file_name).stem[:60]}"
            labeled_text = normalize_whitespace(f"""
Document_Type = {doc_type},
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {number},
Level = {level},
Source_Page = ,
Linked_File_or_Page = {linked},
File_Name = {file_name},
Content = {content}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": doc_type,
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": number,
                "level": level,
                "source_page": "",
                "linked_file_or_page": linked,
                "file_name": file_name,
                "text": labeled_text,
            })
    else:
        print(f"Warning: PDF folder was not found: {pdf_dir}")

    corpus_df = pd.DataFrame(rows)
    if corpus_df.empty:
        raise ValueError("No department documents were found. Check SOURCE_ROOT and PDF_FOLDER_NAME.")

    corpus_path = data_dir / "corpus.csv"
    corpus_df.to_csv(corpus_path, index=False)
    print(f"Department corpus saved to: {corpus_path}")
    print(f"Documents created: {len(corpus_df)}")
    print(corpus_df["document_type"].value_counts())
    return corpus_df


def load_corpus(corpus_name="corpus", rebuild=False):
    corpus_fullcsvname = DATA_DIR / f"{corpus_name}.csv"

    if rebuild or not corpus_fullcsvname.exists():
        print("Building a new CS/EE department corpus...")
        corpus_df = build_department_corpus()
    else:
        corpus_df = pd.read_csv(corpus_fullcsvname).fillna("")

    print(f"Corpus: {len(corpus_df)} documents")
    print(f"Columns: {list(corpus_df.columns)}")
    print()
    print("First 3 documents:")
    for _, row in corpus_df.head(3).iterrows():
        preview = str(row["text"])[:220] + "..." if len(str(row["text"])) > 220 else str(row["text"])
        print(f"  {row['doc_id']}: {preview}")
    print()

    return corpus_df


In [3]:
corpus_name = "corpus"

# Use True the first time you switch from the old healthcare corpus to the CS/EE corpus.
# After data/corpus.csv has been created, you can set this to False for faster reruns.
REBUILD_CORPUS = True

corpus_df = load_corpus(corpus_name, rebuild=REBUILD_CORPUS)


Building a new CS/EE department corpus...
Using existing source folder: C:\Users\xande\Projects_Py\EE\eecs_search_review_update\DepCourseInfo
Reading 266 PDFs from DepCourseInfo\downloaded_forms_and_worksheets ...
Department corpus saved to: data\corpus.csv
Documents created: 532
document_type
course_listing               275
course_syllabus              173
program_form_or_worksheet     74
department_document           10
Name: count, dtype: int64
Corpus: 532 documents
Columns: ['doc_id', 'document_type', 'title', 'course_code', 'prefix', 'number', 'level', 'source_page', 'linked_file_or_page', 'file_name', 'text']

First 3 documents:
  course_0001_CAP_5615: Document_Type = course_listing, Title = Intro to Neural Networks, Course_Code = CAP 5615, Prefix = CAP, Number = 5615, Level = graduate, Source_Page = https://www.fau.edu/engineering/eecs/research/nrt/faculty/collaborato...
  course_0002_CAP_5615: Document_Type = course_listing, Title = Introduction to Neural Networks, Course_Code

## Chunked Retrieval Corpus

This block keeps the original document-level `corpus_df`, but creates a second retrieval corpus named `active_corpus_df` / `retrieval_df`.

BM25 and FAISS should index `active_corpus_df`, where each row is an overlapping chunk from a source document. This lets hybrid search retrieve specific parts of longer worksheets, forms, PDFs, and course lists instead of matching only document headings or filenames.

In [4]:
# ============================================================
# Build a chunked retrieval corpus for BM25 + FAISS
# ============================================================
# Why this exists:
# - corpus_df keeps one row per original source document.
# - retrieval_df / active_corpus_df keeps many smaller chunks per document.
# - BM25 and FAISS index the chunks, which improves recall for long PDFs/forms.
# - Evaluation qrels are expanded from original document IDs to chunk IDs later.

CHUNKED_CORPUS_NAME = "corpus_chunks"
REBUILD_RETRIEVAL_CHUNKS = True
CHUNK_SIZE_WORDS = 850
CHUNK_OVERLAP_WORDS = 150
MIN_CHUNK_WORDS = 35


def extract_content_from_labeled_text(text):
    """Return the Content field from the labeled corpus text when possible."""
    text = normalize_whitespace(text)
    match = re.search(r"\bContent\s*=\s*(.*)$", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        content = match.group(1).strip(" ,")
        if content:
            return content
    return text


def chunk_words_with_overlap(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS, min_words=MIN_CHUNK_WORDS):
    """Split text into overlapping word chunks."""
    text = normalize_whitespace(text)
    words = text.split()

    if not words:
        return []

    if len(words) <= chunk_size:
        return [" ".join(words)] if len(words) >= min_words or len(words) > 0 else []

    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk = " ".join(words[start:end]).strip()

        # Avoid adding tiny tail chunks unless this is the only chunk.
        if len(chunk.split()) >= min_words:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks


def looks_like_low_value_repetition(text, max_repetition_ratio=0.35):
    """Filter chunks that are mostly repeated OCR/PDF noise, such as 'phd phd phd ...'."""
    words = re.findall(r"[A-Za-z]+", str(text).lower())
    if len(words) < 40:
        return False
    counts = Counter(words)
    most_common_word, most_common_count = counts.most_common(1)[0]
    return most_common_count / max(len(words), 1) > max_repetition_ratio


def metadata_prefix_for_chunk(row, chunk_index, chunk_count):
    """Attach enough metadata for retrieval without letting metadata overwhelm the chunk text."""
    title = str(row.get("title", "")).strip()
    document_type = str(row.get("document_type", "")).strip()
    course_code = str(row.get("course_code", "")).strip()
    prefix = str(row.get("prefix", "")).strip()
    number = str(row.get("number", "")).strip()
    level = str(row.get("level", "")).strip()
    file_name = str(row.get("file_name", "")).strip()
    source_page = str(row.get("source_page", "")).strip()
    linked = str(row.get("linked_file_or_page", "")).strip()
    parent_doc_id = str(row.get("doc_id", "")).strip()

    return normalize_whitespace(f"""
Document_Type = {document_type},
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {number},
Level = {level},
Parent_Doc_ID = {parent_doc_id},
Chunk_Index = {chunk_index} of {chunk_count},
Source_Page = {source_page},
Linked_File_or_Page = {linked},
File_Name = {file_name},
Content =
""")


def build_chunked_retrieval_corpus(corpus_df, data_dir=DATA_DIR):
    """
    Create chunk-level rows for retrieval.

    Important columns:
    - doc_id: unique chunk ID used by BM25/FAISS
    - parent_doc_id: original source document ID
    - text: metadata + chunk text used for indexing
    - chunk_text: just the chunk body used for RAG context compression
    """
    rows = []

    for _, row in corpus_df.fillna("").iterrows():
        parent_doc_id = str(row.get("doc_id", "")).strip()
        full_text = str(row.get("text", ""))
        content = extract_content_from_labeled_text(full_text)
        chunks = chunk_words_with_overlap(content)

        if not chunks:
            chunks = [normalize_whitespace(full_text)]

        chunk_count = len(chunks)
        for chunk_index, chunk_text in enumerate(chunks, start=1):
            if looks_like_low_value_repetition(chunk_text):
                continue

            chunk_id = f"{parent_doc_id}::chunk_{chunk_index:03d}"
            prefix = metadata_prefix_for_chunk(row, chunk_index, chunk_count)
            index_text = normalize_whitespace(prefix + " " + chunk_text)

            rows.append({
                "doc_id": chunk_id,
                "parent_doc_id": parent_doc_id,
                "chunk_index": chunk_index,
                "chunk_count": chunk_count,
                "chunk_text": normalize_whitespace(chunk_text),
                "text": index_text,
                "document_type": str(row.get("document_type", "")).strip(),
                "title": str(row.get("title", "")).strip(),
                "course_code": str(row.get("course_code", "")).strip(),
                "prefix": str(row.get("prefix", "")).strip(),
                "number": str(row.get("number", "")).strip(),
                "level": str(row.get("level", "")).strip(),
                "source_page": str(row.get("source_page", "")).strip(),
                "linked_file_or_page": str(row.get("linked_file_or_page", "")).strip(),
                "file_name": str(row.get("file_name", "")).strip(),
            })

    retrieval_df = pd.DataFrame(rows)
    if retrieval_df.empty:
        raise ValueError("No retrieval chunks were created. Check corpus_df and chunking settings.")

    output_path = data_dir / f"{CHUNKED_CORPUS_NAME}.csv"
    retrieval_df.to_csv(output_path, index=False)

    print(f"Chunked retrieval corpus saved to: {output_path}")
    print(f"Original source documents: {len(corpus_df)}")
    print(f"Retrieval chunks created: {len(retrieval_df)}")
    print(f"Average chunks per source document: {len(retrieval_df) / max(len(corpus_df), 1):.2f}")
    print("Chunk length summary, in words:")
    print(retrieval_df["chunk_text"].astype(str).apply(lambda x: len(x.split())).describe())

    return retrieval_df


def load_or_build_retrieval_chunks(corpus_df, rebuild=REBUILD_RETRIEVAL_CHUNKS):
    chunk_path = DATA_DIR / f"{CHUNKED_CORPUS_NAME}.csv"

    if rebuild or not chunk_path.exists():
        retrieval_df = build_chunked_retrieval_corpus(corpus_df)
    else:
        retrieval_df = pd.read_csv(chunk_path).fillna("")

    print(f"Active retrieval corpus: {len(retrieval_df)} chunks")
    print(f"Columns: {list(retrieval_df.columns)}")
    print("\nFirst 3 retrieval chunks:")
    for _, row in retrieval_df.head(3).iterrows():
        preview = str(row["chunk_text"])[:220] + "..." if len(str(row["chunk_text"])) > 220 else str(row["chunk_text"])
        print(f"  {row['doc_id']}  | parent={row.get('parent_doc_id', '')}")
        print(f"    {preview}")
    print()

    return retrieval_df


retrieval_df = load_or_build_retrieval_chunks(corpus_df)

# Use active_corpus_df for retrieval, indexing, and RAG context.
# Keep corpus_df unchanged for full-document metadata and inspection.
active_corpus_df = retrieval_df
RETRIEVAL_IS_CHUNKED = "parent_doc_id" in active_corpus_df.columns


Chunked retrieval corpus saved to: data\corpus_chunks.csv
Original source documents: 532
Retrieval chunks created: 569
Average chunks per source document: 1.07
Chunk length summary, in words:
count    569.000000
mean     192.792619
std      244.669832
min        2.000000
25%        5.000000
50%      142.000000
75%      269.000000
max      850.000000
Name: chunk_text, dtype: float64
Active retrieval corpus: 569 chunks
Columns: ['doc_id', 'parent_doc_id', 'chunk_index', 'chunk_count', 'chunk_text', 'text', 'document_type', 'title', 'course_code', 'prefix', 'number', 'level', 'source_page', 'linked_file_or_page', 'file_name']

First 3 retrieval chunks:
  course_0001_CAP_5615::chunk_001  | parent=course_0001_CAP_5615
    Instructor for CAP 5615 Intro to Neural Networks
  course_0002_CAP_5615::chunk_001  | parent=course_0002_CAP_5615
    CAP 5615 Introduction to Neural Networks
  course_0003_CAP_5625::chunk_001  | parent=course_0003_CAP_5625
    CAP 5625



In [5]:
DEFAULT_DEPARTMENT_QUERIES = [('q01', 'Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.'), ('q02', 'Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.'), ('q03', 'Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.'), ('q04', 'Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.'), ('q05', 'Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.'), ('q06', 'Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering graduate program information, and electrical engineering degree requirements.'), ('q07', 'Find cybersecurity certificate and minor documents, cyber security worksheets, cybersecurity applications, and security-related courses such as cryptography, network security, and data security.'), ('q08', 'Find data science and analytics certificate documents, DSA concentration worksheets, CSDA or computer science data analytics program sheets, and related data science courses.'), ('q09', 'Find machine learning, artificial intelligence, neural networks, deep learning, computer vision, and NLP course syllabi.'), ('q10', 'Find data mining, information retrieval, web mining, big data analytics, and database-related course syllabi.'), ('q11', 'Find software engineering courses and syllabi covering software requirements engineering, software testing, software architecture, software maintenance, and object-oriented design.'), ('q12', 'Find electrical engineering course syllabi for circuits, electronics, signal processing, communications, control systems, antennas, RF, power systems, and smart grid topics.'), ('q13', 'Find computer architecture, digital logic, microprocessor, embedded systems, VLSI, and computer design course syllabi.'), ('q14', 'Find general EECS student forms and administrative documents such as audit forms, application forms, graduate pathway scholarships, low residency PhD forms, and conflict of interest disclosure forms.')]


def create_default_queries(query_path):
    query_path = Path(query_path)
    query_path.parent.mkdir(parents=True, exist_ok=True)
    query_df = pd.DataFrame(DEFAULT_DEPARTMENT_QUERIES, columns=["query_id", "query_text"])
    query_df.to_csv(query_path, index=False)
    print(f"Default CS/EE department queries saved to: {query_path}")
    return query_df


def load_premade_queries(query_path=None, rebuild=False):
    """Load the starter EECS query list from search_review/eecs_queries.csv."""
    query_path = Path(query_path) if query_path is not None else SEARCH_REVIEW_DIR / "eecs_queries.csv"

    if rebuild or not query_path.exists():
        queries_df = create_default_queries(query_path)
    else:
        queries_df = pd.read_csv(query_path)

    print(f"Queries loaded from: {query_path}")
    print(f"Queries: {len(queries_df)} queries")
    for _, row in queries_df.iterrows():
        print(f"  {row['query_id']}: {row['query_text']}")
    return queries_df


QUERY_PATH = SEARCH_REVIEW_DIR / "eecs_queries.csv"

# Keep False when using the reviewed starter CSVs in search_review/.
# Set True only when you intentionally want to regenerate the starter query list.
REBUILD_QUERIES = False

queries_df = load_premade_queries(QUERY_PATH, rebuild=REBUILD_QUERIES)


Queries loaded from: search_review\eecs_queries.csv
Queries: 14 queries
  q01: Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.
  q02: Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.
  q03: Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.
  q04: Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.
  q05: Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.
  q06: Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering gradua

In [6]:
QUERY_PATH = SEARCH_REVIEW_DIR / "eecs_queries.csv"

# Keep False when using the reviewed starter CSVs in search_review/.
# Set True only when you intentionally want to regenerate the starter query list.
REBUILD_QUERIES = False

queries_df = load_premade_queries(QUERY_PATH, rebuild=REBUILD_QUERIES)


Queries loaded from: search_review\eecs_queries.csv
Queries: 14 queries
  q01: Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements.
  q02: Find artificial intelligence minor forms, AI minor worksheets, minor program applications, and requirements for students completing an AI minor.
  q03: Find big data analytics certificate forms, professional big data analytics certificate worksheets, applications, and related big data analytics program requirements.
  q04: Find MS Computer Science program worksheets, MS CSE worksheet documents, professional MS CS worksheets, and computer science graduate program requirements.
  q05: Find MS Computer Engineering program sheets, MS COEN worksheets, computer engineering application forms, and computer engineering graduate requirements.
  q06: Find MS Electrical Engineering worksheets, EEL program sheets, electrical engineering gradua

In [7]:
# Dense embedding model used by FAISS.
# BGE-M3 is the retrieval upgrade for this notebook.
# It improves the dense FAISS retrieval side; BM25 still handles the sparse keyword side.
MODEL_NAME = globals().get("MODEL_NAME", "BAAI/bge-m3")

# BGE-M3 is larger than MiniLM, so a smaller batch size is safer on laptops/CPU.
# Increase this if you have a strong GPU.
EMBED_BATCH_SIZE = globals().get(
    "EMBED_BATCH_SIZE",
    16 if "bge-m3" in MODEL_NAME.lower() else 64
)

print(f"Loading embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Embedding batch size: {EMBED_BATCH_SIZE}")


Loading embedding model: BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 1024
Embedding batch size: 16


In [8]:
def faiss_embedding_process(corpus_df):
    doc_ids = corpus_df['doc_id'].tolist()
    doc_texts = corpus_df['text'].astype(str).tolist()
    
    print(f"Encoding {len(doc_texts)} documents...")
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2 normalize for cosine similarity
    )
    
    print(f"Embedding shape: {doc_embeddings.shape}")
    print(f"Sample vector (first 10 dims): {doc_embeddings[0][:10]}")

    return doc_ids, doc_texts, doc_embeddings
    
    

In [9]:
doc_ids, doc_texts, doc_embeddings = faiss_embedding_process(active_corpus_df)

Encoding 569 documents...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Embedding shape: (569, 1024)
Sample vector (first 10 dims): [-0.04400833 -0.0446574   0.00981059 -0.00244941 -0.01062262  0.01916227
  0.01983805  0.02718064 -0.02253107  0.04992457]


In [10]:
def faiss_index_process(doc_embeddings):
    dim = doc_embeddings.shape[1]  # 384
    index = faiss.IndexFlatIP(dim)
    index.add(doc_embeddings.astype(np.float32))
    
    print(f"FAISS index built: {index.ntotal} vectors, dimension={dim}")

    return index

In [11]:
index = faiss_index_process(doc_embeddings)

FAISS index built: 569 vectors, dimension=1024


In [12]:
def dense_search(query_text, model, index, doc_ids, top_k=10):
    """
    Encode a query and return the top-k most similar documents.

    Returns: list of (doc_id, score, rank) tuples
    """
    # Encode the query with the SAME model and normalization
    query_vec = model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    # Search the FAISS index
    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append((doc_ids[idx], float(score), rank))

    return results

In [13]:
test_query = queries_df.iloc[0]
print(f"Query: {test_query['query_id']} — \"{test_query['query_text']}\"\n")

results = dense_search(test_query['query_text'], model, index, doc_ids, top_k=5)

for doc_id, score, rank in results:
    doc_text = active_corpus_df[active_corpus_df['doc_id'] == doc_id]['text'].values[0]
    preview = str(doc_text)[:200] + '...' if len(str(doc_text)) > 200 else str(doc_text)
    print(f"  Rank {rank} | Score: {score:.4f} | {doc_id}")
    print(f"    {preview}\n")

Query: q01 — "Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements."

  Rank 1 | Score: 0.7207 | pdf_0281_artificial-intelligence-certificate-program-application-jan6::chunk_001
    Document_Type = program_form_or_worksheet, Title = Artificial Intelligence Certificate Program Application Jan6, Course_Code = , Prefix = , Number = , Level = , Parent_Doc_ID = pdf_0281_artificial-int...

  Rank 2 | Score: 0.7199 | pdf_0513_professional-ai-certificate-program-application::chunk_001
    Document_Type = program_form_or_worksheet, Title = Professional Ai Certificate Program Application, Course_Code = , Prefix = , Number = , Level = , Parent_Doc_ID = pdf_0513_professional-ai-certificate...

  Rank 3 | Score: 0.7100 | pdf_0514_professional-certificate-ai-worksheet::chunk_001
    Document_Type = program_form_or_worksheet, Title = Professional Certificate Ai Worksheet, Course_Cod

In [14]:
TOP_K = 100
all_results = []

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    results = dense_search(qtext, model, index, doc_ids, top_k=TOP_K)
    for doc_id, score, rank in results:
        all_results.append({
            'query_id': qid,
            'doc_id': doc_id,
            'rank': rank,
            'score': round(score, 4),
        })

run_dense_df = pd.DataFrame(all_results)
run_dense_df.to_csv('outputs/run_dense.csv', index=False)

print(f"Dense retrieval results saved: {len(run_dense_df)} rows")
print(f"Queries processed: {run_dense_df['query_id'].nunique()}")
run_dense_df.head(20)

Dense retrieval results saved: 1400 rows
Queries processed: 14


,query_id,doc_id,rank,score
0,q01,pdf_0281_artificial-intelligence-certificate-p...,1,0.7207
1,q01,pdf_0513_professional-ai-certificate-program-a...,2,0.7199
2,q01,pdf_0514_professional-certificate-ai-worksheet...,3,0.7100
3,q01,pdf_0341_certificate-artificial--intelligence-...,4,0.7076
4,q01,pdf_0343_certificate-artificial-intelligence-p...,5,0.6876
5,q01,pdf_0342_certificate-artificial-intelligence--...,6,0.6865
6,q01,pdf_0278_artificial-inteligence-certificate-wo...,7,0.6691
7,q01,course_0025_CAP_6635::chunk_001,8,0.6553
8,q01,pdf_0510_newcertificate-artificial-intelligenc...,9,0.6542
9,q01,pdf_0280_artificial-intelligence-cert-program-...,10,0.6520


In [15]:
def tokenize_for_bm25(text):
    words = re.findall(r"[A-Za-z0-9]+", str(text).lower())
    return [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]


def bm25_tokenize_corpus(corpus_df):
    tokenized_corpus = [tokenize_for_bm25(doc) for doc in corpus_df["text"]]
    return tokenized_corpus


def bm25_corpus(tokenized_corpus):
    bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.8)
    return bm25


In [16]:
tokenized_corpus = bm25_tokenize_corpus(active_corpus_df)
bm25 = bm25_corpus(tokenized_corpus)

In [17]:
bm25_results = []
for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = tokenize_for_bm25(qrow['query_text'])
    scores = bm25.get_scores(qtext)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_indices, start=1):
        bm25_results.append({
            'query_id': qid,
            'doc_id': doc_ids[idx],
            'rank': rank,
            'score': round(float(scores[idx]), 4),
        })

run_bm25_df = pd.DataFrame(bm25_results)
run_bm25_df.to_csv('outputs/run_bm25.csv', index=False)
print(f"BM25 results: {len(run_bm25_df)} rows")
run_bm25_df.head(10)


BM25 results: 1400 rows


,query_id,doc_id,rank,score
0,q01,pdf_0513_professional-ai-certificate-program-a...,1,46.6874
1,q01,pdf_0514_professional-certificate-ai-worksheet...,2,45.4531
2,q01,pdf_0341_certificate-artificial--intelligence-...,3,34.8508
3,q01,pdf_0281_artificial-intelligence-certificate-p...,4,34.6511
4,q01,pdf_0483_minor--artificial-intelligence-applic...,5,34.6312
5,q01,pdf_0279_artificial-inteligence-minor-workshee...,6,34.2977
6,q01,pdf_0515_professional-certificate-big-data-ana...,7,34.1373
7,q01,pdf_0282_artificial-intelligence-minor-program...,8,33.9722
8,q01,pdf_0516_professional-certificate-big-data-ana...,9,33.4395
9,q01,pdf_0343_certificate-artificial-intelligence-p...,10,32.9746


In [18]:
comparison_rows = []

for qid in queries_df['query_id']:
    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    for rank in range(1, 6):
        bm25_row = bm25_q[bm25_q['rank'] == rank]
        dense_row = dense_q[dense_q['rank'] == rank]

        comparison_rows.append({
            'query_id': qid,
            'rank': rank,
            'bm25_doc_id': bm25_row['doc_id'].values[0] if len(bm25_row) > 0 else '',
            'bm25_score': round(bm25_row['score'].values[0], 4) if len(bm25_row) > 0 else '',
            'dense_doc_id': dense_row['doc_id'].values[0] if len(dense_row) > 0 else '',
            'dense_score': round(dense_row['score'].values[0], 4) if len(dense_row) > 0 else '',
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('outputs/comparison.csv', index=False)

print("Side-by-side comparison saved to comparison.csv")
comparison_df.head(10)

Side-by-side comparison saved to comparison.csv


,query_id,rank,bm25_doc_id,bm25_score,dense_doc_id,dense_score
0,q01,1,pdf_0513_professional-ai-certificate-program-a...,46.6874,pdf_0281_artificial-intelligence-certificate-p...,0.7207
1,q01,2,pdf_0514_professional-certificate-ai-worksheet...,45.4531,pdf_0513_professional-ai-certificate-program-a...,0.7199
2,q01,3,pdf_0341_certificate-artificial--intelligence-...,34.8508,pdf_0514_professional-certificate-ai-worksheet...,0.7100
3,q01,4,pdf_0281_artificial-intelligence-certificate-p...,34.6511,pdf_0341_certificate-artificial--intelligence-...,0.7076
4,q01,5,pdf_0483_minor--artificial-intelligence-applic...,34.6312,pdf_0343_certificate-artificial-intelligence-p...,0.6876
5,q02,1,pdf_0483_minor--artificial-intelligence-applic...,45.1198,pdf_0282_artificial-intelligence-minor-program...,0.7636
6,q02,2,pdf_0282_artificial-intelligence-minor-program...,44.9593,pdf_0483_minor--artificial-intelligence-applic...,0.7550
7,q02,3,pdf_0279_artificial-inteligence-minor-workshee...,40.6432,pdf_0279_artificial-inteligence-minor-workshee...,0.7189
8,q02,4,pdf_0342_certificate-artificial-intelligence--...,37.8315,pdf_0342_certificate-artificial-intelligence--...,0.7001
9,q02,5,pdf_0343_certificate-artificial-intelligence-p...,37.1018,pdf_0281_artificial-intelligence-certificate-p...,0.6934


In [19]:
doc_lookup = dict(zip(active_corpus_df['doc_id'], active_corpus_df['text'].astype(str)))

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    print(f"\n{'='*80}")
    print(f"Query {qid}: \"{qtext}\"")
    print(f"{'='*80}")

    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    print(f"\n{'BM25 Results':<40} | {'Dense Retrieval Results':<40}")
    print(f"{'-'*40} | {'-'*40}")

    for rank in range(5):
        # BM25 side
        if rank < len(bm25_q):
            b = bm25_q.iloc[rank]
            b_text = doc_lookup.get(b['doc_id'], '')[:60]
            bm25_str = f"#{b['rank']} {b['doc_id']} ({b['score']:.2f}) {b_text}"
        else:
            bm25_str = ""

        # Dense side
        if rank < len(dense_q):
            d = dense_q.iloc[rank]
            d_text = doc_lookup.get(d['doc_id'], '')[:60]
            dense_str = f"#{d['rank']} {d['doc_id']} ({d['score']:.4f}) {d_text}"
        else:
            dense_str = ""

        print(f"{bm25_str:<40} | {dense_str:<40}")

    # Overlap analysis
    bm25_docs = set(bm25_q['doc_id'])
    dense_docs = set(dense_q['doc_id'])
    overlap = bm25_docs & dense_docs
    print(f"\nOverlap: {len(overlap)} / 5 docs appear in both top-5")
    if overlap:
        print(f"  Shared docs: {overlap}")
    print(f"  BM25-only: {bm25_docs - dense_docs}")
    print(f"  Dense-only: {dense_docs - bm25_docs}")


Query q01: "Find artificial intelligence certificate forms, certificate worksheets, program applications, professional AI certificate documents, and related AI certificate requirements."

BM25 Results                             | Dense Retrieval Results                 
---------------------------------------- | ----------------------------------------
#1 pdf_0513_professional-ai-certificate-program-application::chunk_001 (46.69) Document_Type = program_form_or_worksheet, Title = Professio | #1 pdf_0281_artificial-intelligence-certificate-program-application-jan6::chunk_001 (0.7207) Document_Type = program_form_or_worksheet, Title = Artificia
#2 pdf_0514_professional-certificate-ai-worksheet::chunk_001 (45.45) Document_Type = program_form_or_worksheet, Title = Professio | #2 pdf_0513_professional-ai-certificate-program-application::chunk_001 (0.7199) Document_Type = program_form_or_worksheet, Title = Professio
#3 pdf_0341_certificate-artificial--intelligence-program-application::chun

In [20]:
#bm25_weight= 2.5, dense_weight= 0.3
def rrf_process(bm25_df, dense_df, k=60, bm25_weight= 1.0 , dense_weight= 1.0):
    fused_results = []

    bm_ids = set(bm25_df["query_id"])
    dense_ids = set(dense_df["query_id"])
    query_ids = sorted(bm_ids.union(dense_ids))

    for qid in query_ids:
        rrf_scores = {}

        bm25_q = bm25_df[bm25_df["query_id"] == qid]
        dense_q = dense_df[dense_df["query_id"] == qid]

        # BM25 contribution
        for _, row in bm25_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                bm25_weight / (k + rank)
            )

        # Dense contribution
        for _, row in dense_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                dense_weight / (k + rank)
            )

        ranked_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        for final_rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            fused_results.append({
                "query_id": qid,
                "doc_id": doc_id,
                "rrf_score": round(score, 6),
                "rank": final_rank
            })

    fused_df = pd.DataFrame(fused_results)
    return fused_df

    

In [21]:
def preprocess_text(text):
    words = re.findall(r"[A-Za-z0-9]+", str(text).lower())
    
    cleaned_words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]
    
    return set(cleaned_words)


def simple_word_match_percent(query_text, doc_text):
    query_words = preprocess_text(query_text)
    doc_words = preprocess_text(doc_text)

    if not query_words:
        return 0.0

    overlap = query_words.intersection(doc_words)
    return (len(overlap) / len(query_words)) * 100


def semantic_similarity_score(query_text, doc_text, embedding_model):
    embeddings = embedding_model.encode(
        [str(query_text), str(doc_text)],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    query_vec = embeddings[0]
    doc_vec = embeddings[1]

    cosine_sim = float(np.dot(query_vec, doc_vec))
    cs = round(cosine_sim, 2)
    return cs


In [22]:
SUMMARY_MODEL_NAME = "google/flan-t5-small"

print(f"Loading summarization model: {SUMMARY_MODEL_NAME}")
summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARY_MODEL_NAME)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARY_MODEL_NAME)
print("Summarization model loaded.")

# Lookup tables for retrieval_id -> chunk text and metadata
active_corpus_df = active_corpus_df.fillna("")
doc_lookup = dict(zip(active_corpus_df["doc_id"], active_corpus_df["text"].astype(str)))
metadata_lookup = active_corpus_df.set_index("doc_id").to_dict(orient="index")

FIELD_NAMES = [
    "Document_Type",
    "Title",
    "Course_Code",
    "Prefix",
    "Number",
    "Level",
    "Source_Page",
    "Linked_File_or_Page",
    "File_Name",
    "Content"
]


def field_to_pattern(field_name):
    """
    Allow matching labels written with underscores or spaces.
    Example: Course_Code or Course Code
    """
    return re.escape(field_name).replace(r"\_", r"[ _]+")


def extract_structured_fields(doc_text):
    """
    Extract the labeled metadata fields from one department corpus document.
    """
    text = " ".join(str(doc_text).replace("\n", " ").split())
    extracted = {}

    for i, field in enumerate(FIELD_NAMES):
        current_label = field_to_pattern(field)

        if i < len(FIELD_NAMES) - 1:
            next_labels = [field_to_pattern(f) for f in FIELD_NAMES[i + 1:]]
            next_label_pattern = "|".join(next_labels)
            pattern = rf"{current_label}\s*=\s*(.*?)(?=,\s*(?:{next_label_pattern})\s*=|$)"
        else:
            pattern = rf"{current_label}\s*=\s*(.*)$"

        match = re.search(pattern, text, flags=re.IGNORECASE)
        extracted[field] = match.group(1).strip(" ,") if match else "Not found"

    return extracted


def summarize_document(query_text, doc_text, tokenizer, summary_model, embedding_model, doc_id=None):
    doc_text = str(doc_text).strip()

    empty_result = {
        "Document Type": "Not found",
        "Title": "Not found",
        "Course Code": "Not found",
        "Level": "Not found",
        "File Name": "Not found",
        "Source": "Not found",
        "Word Match %": 0.0,
        "Semantic Similarity": 0.0,
        "Content Summary": "No department document content available."
    }

    if not doc_text:
        return empty_result

    extracted = extract_structured_fields(doc_text)
    content_text = extracted.get("Content", "")
    if not content_text or content_text == "Not found":
        content_text = doc_text

    word_match = simple_word_match_percent(query_text, doc_text)
    semantic_sim = semantic_similarity_score(query_text, doc_text, embedding_model)

    prompt = f"""
Summarize this Computer Science and Electrical Engineering department document in 1 to 2 sentences.

Requirements:
- Focus on forms, worksheets, certificates, degree requirements, course syllabi, prerequisites, course topics, or program information.
- Explain why the document may be relevant to the query.
- Do not use healthcare or patient language.

Metadata:
Document Type = {extracted['Document_Type']}
Title = {extracted['Title']}
Course Code = {extracted['Course_Code']}
Level = {extracted['Level']}
File Name = {extracted['File_Name']}

Query:
{query_text}

Document content:
{content_text[:2500]}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = summary_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=False
    )

    content_summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    if not content_summary:
        content_summary = "No concise summary was generated for this document."

    source_value = extracted["Linked_File_or_Page"]
    if source_value == "Not found" or not source_value:
        source_value = extracted["Source_Page"]

    return {
        "Document Type": extracted["Document_Type"],
        "Title": extracted["Title"],
        "Course Code": extracted["Course_Code"],
        "Level": extracted["Level"],
        "File Name": extracted["File_Name"],
        "Source": source_value if source_value else "Not found",
        "Word Match %": round(word_match, 2),
        "Semantic Similarity": round(semantic_sim, 4),
        "Content Summary": content_summary
    }


Loading summarization model: google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Summarization model loaded.


In [23]:
run_rrf_df = rrf_process(run_bm25_df, run_dense_df, k=60)

# Save fused results
run_rrf_df.to_csv("outputs/run_rrf.csv", index=False)

# Print top 5 per query with summaries
for qid in run_rrf_df["query_id"].unique():
    print(f"\n{'='*80}")
    print(f"RRF Results for Query {qid}")
    print(f"{'='*80}")

    query_matches = queries_df.loc[queries_df["query_id"] == qid, "query_text"]
    if query_matches.empty:
        print(f"No query text found for {qid}")
        continue

    query_text = query_matches.iloc[0]
    top_5 = run_rrf_df[run_rrf_df["query_id"] == qid].head(5)

    for _, row in top_5.iterrows():
        doc_id = row["doc_id"]
        doc_text = doc_lookup.get(doc_id, "")

        doc_summary = summarize_document(
            query_text=query_text,
            doc_text=doc_text,
            tokenizer=summary_tokenizer,
            summary_model=summary_model,
            embedding_model=model,
            doc_id=doc_id
        )

        print(
            f"Rank #{row['rank']:>2} | "
            f"Doc: {doc_id} | "
            f"RRF Score: {row['rrf_score']:.6f}"
        )
        print(f"Title: {doc_summary['Title']}")
        print(f"Type: {doc_summary['Document Type']} | Course: {doc_summary['Course Code']} | Level: {doc_summary['Level']}")
        print(f"Word Match: {doc_summary['Word Match %']}% | Semantic Similarity: {doc_summary['Semantic Similarity']}")
        print(f"Summary: {doc_summary['Content Summary']}")
        print("-" * 80)



RRF Results for Query q01
Rank # 1 | Doc: pdf_0513_professional-ai-certificate-program-application::chunk_001 | RRF Score: 0.032522
Title: Professional Ai Certificate Program Application
Type: program_form_or_worksheet | Course:  | Level: Parent_Doc_ID = pdf_0513_professional-ai-certificate-program-application, Chunk_Index = 1 of 1
Word Match: 84.62% | Semantic Similarity: 0.72
Summary: Applicants for the Artificial Intelligence Certificate Program must submit a cover letter to the program.
--------------------------------------------------------------------------------
Rank # 2 | Doc: pdf_0281_artificial-intelligence-certificate-program-application-jan6::chunk_001 | RRF Score: 0.032018
Title: Artificial Intelligence Certificate Program Application Jan6
Type: program_form_or_worksheet | Course:  | Level: Parent_Doc_ID = pdf_0281_artificial-intelligence-certificate-program-application-jan6, Chunk_Index = 1 of 1
Word Match: 69.23% | Semantic Similarity: 0.72
Summary: College of Engineer

In [24]:
qrels_path = SEARCH_REVIEW_DIR / "eecs_qrels.csv"
qrels_available = qrels_path.exists()

if qrels_available:
    qrels_original_df = pd.read_csv(qrels_path).fillna("")
    qrels_df = qrels_original_df.copy()

    # If retrieval is chunked, qrels written for original source documents need to be expanded
    # so evaluation compares retrieved chunk IDs against chunk-level relevance labels.
    if globals().get("RETRIEVAL_IS_CHUNKED", False) and "parent_doc_id" in active_corpus_df.columns:
        chunk_map = active_corpus_df[["doc_id", "parent_doc_id"]].copy()
        chunk_map = chunk_map.rename(columns={"doc_id": "chunk_doc_id"})

        expanded_qrels = qrels_original_df.merge(
            chunk_map,
            left_on="doc_id",
            right_on="parent_doc_id",
            how="inner"
        )

        if not expanded_qrels.empty:
            qrels_df = (
                expanded_qrels[["query_id", "chunk_doc_id", "relevance"]]
                .rename(columns={"chunk_doc_id": "doc_id"})
                .drop_duplicates()
                .reset_index(drop=True)
            )
            print(
                f"Expanded qrels from {len(qrels_original_df)} document-level rows "
                f"to {len(qrels_df)} chunk-level rows for evaluation."
            )
        else:
            print("Warning: qrels could not be expanded to chunk IDs. Metrics may be zero if IDs do not overlap.")
else:
    qrels_original_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])
    qrels_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])
    print("No search_review/eecs_qrels.csv file was found. Skipping supervised retrieval evaluation metrics.")
    print("Retrieval still works; qrels are only needed for P@K, MAP, and nDCG scoring.")


def precision_at_k(ranked_doc_ids, rel_map, k=10):
    top = ranked_doc_ids[:k]
    if len(top) == 0:
        return 0.0
    hits = sum(1 for d in top if rel_map.get(d, 0) > 0)  # relevance 1 or 2 counts as relevant
    return hits / len(top)

# Build relevance lookup: qid -> {doc_id: relevance}
rel_by_q = {}
if qrels_available:
    for qid, g in qrels_df.groupby("query_id"):
        rel_by_q[str(qid)] = {
            str(r.doc_id): int(r.relevance)
            for r in g.itertuples(index=False)
        }


def evaluate_p10(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping P@{k} for {run_name}; qrels are not available.")
        return 0.0

    p10_values = []

    print(f"\nP@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        p10 = precision_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        p10_values.append(p10)
        print(f"{qid}: {p10:.3f}")

    mean_p10 = float(np.mean(p10_values)) if p10_values else 0.0
    print(f"Mean P@{k} for {run_name}: {mean_p10:.3f}")
    return mean_p10

# Evaluate BM25
mean_p10_bm25 = evaluate_p10(run_bm25_df, "BM25", k=10)

# Evaluate FAISS / Dense retrieval
mean_p10_faiss = evaluate_p10(run_dense_df, "FAISS Dense Search", k=10)

# Evaluate RRF
mean_p10_rrf = evaluate_p10(run_rrf_df, "RRF Fusion", k=10)


Expanded qrels from 388 document-level rows to 180 chunk-level rows for evaluation.

P@10 per query (BM25):
q01: 0.500
q02: 0.400
q03: 0.400
q04: 0.000
q05: 0.000
q06: 0.000
q07: 0.000
q08: 0.300
q09: 0.200
q10: 0.100
q11: 0.700
q12: 0.000
q13: 0.600
q14: 0.400
Mean P@10 for BM25: 0.257

P@10 per query (FAISS Dense Search):
q01: 0.600
q02: 0.600
q03: 0.300
q04: 0.000
q05: 0.000
q06: 0.200
q07: 0.200
q08: 0.500
q09: 0.600
q10: 0.500
q11: 0.900
q12: 0.100
q13: 0.900
q14: 0.000
Mean P@10 for FAISS Dense Search: 0.386

P@10 per query (RRF Fusion):
q01: 0.700
q02: 0.600
q03: 0.300
q04: 0.000
q05: 0.000
q06: 0.100
q07: 0.100
q08: 0.300
q09: 0.600
q10: 0.300
q11: 1.000
q12: 0.100
q13: 0.800
q14: 0.200
Mean P@10 for RRF Fusion: 0.364


In [25]:
print("\n" + "=" * 50)
print("Mean P@10 Summary")
print("=" * 50)
print(f"BM25:              {mean_p10_bm25:.3f}")
print(f"FAISS Dense:       {mean_p10_faiss:.3f}")
print(f"RRF Fusion:        {mean_p10_rrf:.3f}")


Mean P@10 Summary
BM25:              0.257
FAISS Dense:       0.386
RRF Fusion:        0.364


In [26]:
def average_precision(ranked_doc_ids, rel_map):
    # AP uses binary relevance: treat relevance 1 or 2 as relevant
    total_relevant = sum(1 for v in rel_map.values() if v > 0)
    if total_relevant == 0:
        return 0.0

    hits = 0
    ap_sum = 0.0

    for i, d in enumerate(ranked_doc_ids, start=1):
        if rel_map.get(d, 0) > 0:
            hits += 1
            ap_sum += hits / i

    return ap_sum / total_relevant


def evaluate_map(run_df, run_name):
    if not qrels_available:
        print(f"Skipping MAP for {run_name}; qrels are not available.")
        return 0.0

    ap_values = []

    print(f"\nAverage Precision (AP) per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ap = average_precision(ranked, rel_by_q.get(str(qid), {}))
        ap_values.append(ap)
        print(f"{qid}: {ap:.3f}")

    map_score = float(np.mean(ap_values)) if len(ap_values) > 0 else 0.0
    print(f"\nMAP ({run_name}): {map_score:.3f}")
    return map_score


# ---- BM25 ----
map_bm25 = evaluate_map(run_bm25_df, "BM25")

# ---- FAISS / Dense ----
map_faiss = evaluate_map(run_dense_df, "FAISS Dense Search")

# ---- RRF ----
map_rrf = evaluate_map(run_rrf_df, "RRF Fusion")


print("\n" + "=" * 50)
print("MAP Summary")
print("=" * 50)
print(f"BM25:         {map_bm25:.3f}")
print(f"FAISS Dense:  {map_faiss:.3f}")
print(f"RRF Fusion:   {map_rrf:.3f}")



Average Precision (AP) per query (BM25):
q01: 0.370
q02: 0.477
q03: 0.525
q04: 0.000
q05: 0.046
q06: 0.075
q07: 0.172
q08: 0.320
q09: 0.225
q10: 0.211
q11: 0.635
q12: 0.048
q13: 0.382
q14: 0.337

MAP (BM25): 0.273

Average Precision (AP) per query (FAISS Dense Search):
q01: 0.524
q02: 0.665
q03: 0.389
q04: 0.010
q05: 0.010
q06: 0.062
q07: 0.181
q08: 0.284
q09: 0.487
q10: 0.531
q11: 0.594
q12: 0.125
q13: 0.691
q14: 0.016

MAP (FAISS Dense Search): 0.326

Average Precision (AP) per query (RRF Fusion):
q01: 0.497
q02: 0.571
q03: 0.410
q04: 0.006
q05: 0.032
q06: 0.077
q07: 0.211
q08: 0.341
q09: 0.386
q10: 0.388
q11: 0.760
q12: 0.114
q13: 0.611
q14: 0.159

MAP (RRF Fusion): 0.326

MAP Summary
BM25:         0.273
FAISS Dense:  0.326
RRF Fusion:   0.326


In [27]:
def dcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = 0.0
    for i, d in enumerate(ranked_doc_ids[:k], start=1):
        gain = rel_map.get(d, 0)  # graded relevance: 0, 1, 2
        dcg += gain / math.log2(i + 1)
    return dcg


def ndcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = dcg_at_k(ranked_doc_ids, rel_map, k)

    # Ideal DCG: sort true relevance scores from highest to lowest
    ideal_gains = sorted(rel_map.values(), reverse=True)
    idcg = 0.0
    for i, gain in enumerate(ideal_gains[:k], start=1):
        idcg += gain / math.log2(i + 1)

    return (dcg / idcg) if idcg > 0 else 0.0


def evaluate_ndcg(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping nDCG@{k} for {run_name}; qrels are not available.")
        return 0.0

    ndcg_values = []

    print(f"\nnDCG@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ndcg = ndcg_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        ndcg_values.append(ndcg)
        print(f"{qid}: {ndcg:.3f}")

    mean_ndcg = float(np.mean(ndcg_values)) if len(ndcg_values) > 0 else 0.0
    print(f"\nMean nDCG@{k} ({run_name}): {mean_ndcg:.3f}")
    return mean_ndcg

# ---- BM25 ----
mean_ndcg_bm25 = evaluate_ndcg(run_bm25_df, "BM25", k=30)

# ---- FAISS / Dense ----
mean_ndcg_faiss = evaluate_ndcg(run_dense_df, "FAISS Dense Search", k=30)

# ---- RRF ----
mean_ndcg_rrf = evaluate_ndcg(run_rrf_df, "RRF Fusion", k=30)

print("\n" + "=" * 50)
print("nDCG@30 Summary")
print("=" * 50)
print(f"BM25:         {mean_ndcg_bm25:.3f}")
print(f"FAISS Dense:  {mean_ndcg_faiss:.3f}")
print(f"RRF Fusion:   {mean_ndcg_rrf:.3f}")



nDCG@30 per query (BM25):
q01: 0.589
q02: 0.653
q03: 0.653
q04: 0.000
q05: 0.046
q06: 0.138
q07: 0.258
q08: 0.434
q09: 0.303
q10: 0.283
q11: 0.675
q12: 0.051
q13: 0.613
q14: 0.479

Mean nDCG@30 (BM25): 0.370

nDCG@30 per query (FAISS Dense Search):
q01: 0.779
q02: 0.852
q03: 0.545
q04: 0.000
q05: 0.098
q06: 0.169
q07: 0.342
q08: 0.479
q09: 0.596
q10: 0.736
q11: 0.715
q12: 0.321
q13: 0.776
q14: 0.052

Mean nDCG@30 (FAISS Dense Search): 0.461

nDCG@30 per query (RRF Fusion):
q01: 0.664
q02: 0.742
q03: 0.605
q04: 0.000
q05: 0.095
q06: 0.211
q07: 0.331
q08: 0.529
q09: 0.478
q10: 0.440
q11: 0.827
q12: 0.202
q13: 0.754
q14: 0.281

Mean nDCG@30 (RRF Fusion): 0.440

nDCG@30 Summary
BM25:         0.370
FAISS Dense:  0.461
RRF Fusion:   0.440


## Expanded Parameter, Chunking, and Reranking Tuning Loop

In [28]:
# ============================================================
# Expanded Chunking + Retrieval + Reranking Tuning Loop
# ============================================================
# Purpose:
# - Try multiple chunk sizes and overlaps.
# - Rebuild BM25 + FAISS retrieval for each chunking setup.
# - Search an expanded BM25/RRF/dense-weight parameter grid.
# - Apply a CrossEncoder reranker after retrieval to the strongest candidates.
# - Print the best chunking + retrieval + reranking parameters at the bottom.
#
# Required earlier notebook objects:
# - corpus_df
# - queries_df with columns: query_id, query_text
# - model: SentenceTransformer embedding model
# - tokenize_for_bm25()
# - average_precision(), ndcg_at_k()
# - extract_content_from_labeled_text(), metadata_prefix_for_chunk(), looks_like_low_value_repetition()
# - DATA_DIR, SEARCH_REVIEW_DIR
#
# Important:
# Reranking is more expensive than first-stage retrieval. This cell therefore uses:
#   Stage 1: broad chunk + BM25 + dense + RRF search without reranking
#   Stage 2: rerank only the strongest Stage 1 configurations

from itertools import product
from pathlib import Path
import time
import math
import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi

# ----------------------------
# Main switches
# ----------------------------
ENABLE_RERANKING = True
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
RERANKER_BATCH_SIZE = 16

# Keep this False for a practical expanded search.
# Set True only if you want to run the much larger exhaustive grid.
TUNING_USE_FULL_SUGGESTED_GRID = False

# Number of non-reranked Stage 1 configurations to send to the reranker stage.
# Increase this if you want a more exhaustive reranking pass.
TUNING_STAGE1_KEEP_TOP_N = 50

# Save detailed per-query scores for the best configuration.
SAVE_BEST_PER_QUERY_DIAGNOSTICS = True

# ----------------------------
# Chunking grid
# ----------------------------
TUNING_CHUNK_SIZE_WORDS_VALUES = [300, 500, 700, 900]
TUNING_CHUNK_OVERLAP_WORDS_VALUES = [50, 100, 150]
TUNING_MIN_CHUNK_WORDS = 35

# ----------------------------
# Retrieval parameter grid
# ----------------------------
if TUNING_USE_FULL_SUGGESTED_GRID:
    # Larger grid based on the expanded ranges suggested earlier.
    # This can take a long time because chunking and embeddings are rebuilt repeatedly.
    TUNING_TOP_K_VALUES = [50, 75, 100, 125, 150, 200, 300]
    TUNING_BM25_K1_VALUES = [1.6, 1.8, 2.0, 2.2, 2.4, 2.6]
    TUNING_BM25_B_VALUES = [0.65, 0.75, 0.85, 0.90, 0.95, 1.0]
    TUNING_RRF_K_VALUES = [30, 60, 90, 120, 150]
    TUNING_BM25_WEIGHT_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]
    TUNING_DENSE_WEIGHT_VALUES = [1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
else:
    # Practical expanded grid focused around the best values already discovered.
    # This still expands beyond the previous search by testing larger top_k, larger dense weights,
    # lower BM25 weights, higher RRF k values, and multiple chunking layouts.
    TUNING_TOP_K_VALUES = [100, 150, 200, 300]
    TUNING_BM25_K1_VALUES = [1.8, 2.2, 2.6]
    TUNING_BM25_B_VALUES = [0.75, 0.90, 1.0]
    TUNING_RRF_K_VALUES = [60, 90, 120, 150]
    TUNING_BM25_WEIGHT_VALUES = [0.0, 0.25, 0.5]
    TUNING_DENSE_WEIGHT_VALUES = [1.5, 2.0, 2.5, 3.0]

# Rerank only the top N candidates from the RRF result.
# Values are tried after Stage 1 finds promising retrieval configurations.
TUNING_RERANK_TOP_N_VALUES = [30, 50, 100]

# Metric settings
TUNING_NDCG_K = 30
TUNING_COMBINED_NDCG_WEIGHT = 0.5
TUNING_COMBINED_MAP_WEIGHT = 0.5

# Output paths
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_RESULTS_PATH = OUTPUT_DIR / "expanded_tuning_stage1_rrf_results.csv"
RERANK_RESULTS_PATH = OUTPUT_DIR / "expanded_tuning_rerank_results.csv"
ALL_RESULTS_PATH = OUTPUT_DIR / "expanded_tuning_all_results.csv"
BEST_SUMMARY_PATH = OUTPUT_DIR / "best_expanded_tuning_summary.csv"
BEST_RUN_PATH = OUTPUT_DIR / "run_best_expanded_tuned.csv"
BEST_RETRIEVAL_CORPUS_PATH = OUTPUT_DIR / "best_tuned_retrieval_corpus.csv"
BEST_PER_QUERY_DIAGNOSTICS_PATH = OUTPUT_DIR / "best_expanded_tuning_per_query_diagnostics.csv"

# ----------------------------
# Load qrels safely
# ----------------------------
qrels_path = SEARCH_REVIEW_DIR / "eecs_qrels.csv"
qrels_available = qrels_path.exists()

if qrels_available:
    qrels_original_df = pd.read_csv(qrels_path).fillna("")
    qrels_original_df["query_id"] = qrels_original_df["query_id"].astype(str)
    qrels_original_df["doc_id"] = qrels_original_df["doc_id"].astype(str)
    qrels_original_df["relevance"] = qrels_original_df["relevance"].astype(int)
else:
    qrels_original_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])


# ============================================================
# Helper functions
# ============================================================
def chunk_words_with_overlap_custom(text, chunk_size, overlap, min_words=35):
    """Split text into overlapping word chunks using the provided chunk size and overlap."""
    text = normalize_whitespace(text)
    words = text.split()

    if not words:
        return []

    if len(words) <= chunk_size:
        return [" ".join(words)] if len(words) >= min_words or len(words) > 0 else []

    chunks = []
    step = max(1, int(chunk_size) - int(overlap))

    for start in range(0, len(words), step):
        end = start + int(chunk_size)
        chunk = " ".join(words[start:end]).strip()

        if len(chunk.split()) >= min_words:
            chunks.append(chunk)

        if end >= len(words):
            break

    return chunks


def build_tuned_retrieval_corpus(corpus_source_df, chunk_size_words, chunk_overlap_words):
    """Build a chunk-level retrieval dataframe for one chunking setup."""
    if int(chunk_overlap_words) >= int(chunk_size_words):
        raise ValueError("chunk_overlap_words must be smaller than chunk_size_words.")

    rows = []

    for _, row in corpus_source_df.fillna("").iterrows():
        parent_doc_id = str(row.get("doc_id", "")).strip()
        full_text = str(row.get("text", ""))
        content = extract_content_from_labeled_text(full_text)

        chunks = chunk_words_with_overlap_custom(
            content,
            chunk_size=int(chunk_size_words),
            overlap=int(chunk_overlap_words),
            min_words=TUNING_MIN_CHUNK_WORDS,
        )

        if not chunks:
            chunks = [normalize_whitespace(full_text)]

        chunk_count = len(chunks)

        for chunk_index, chunk_text in enumerate(chunks, start=1):
            if looks_like_low_value_repetition(chunk_text):
                continue

            chunk_id = f"{parent_doc_id}::chunk_{chunk_index:03d}"
            prefix = metadata_prefix_for_chunk(row, chunk_index, chunk_count)
            index_text = normalize_whitespace(prefix + " " + chunk_text)

            rows.append({
                "doc_id": chunk_id,
                "parent_doc_id": parent_doc_id,
                "chunk_index": chunk_index,
                "chunk_count": chunk_count,
                "chunk_size_words": int(chunk_size_words),
                "chunk_overlap_words": int(chunk_overlap_words),
                "chunk_text": normalize_whitespace(chunk_text),
                "text": index_text,
                "document_type": str(row.get("document_type", "")).strip(),
                "title": str(row.get("title", "")).strip(),
                "course_code": str(row.get("course_code", "")).strip(),
                "prefix": str(row.get("prefix", "")).strip(),
                "number": str(row.get("number", "")).strip(),
                "level": str(row.get("level", "")).strip(),
                "source_page": str(row.get("source_page", "")).strip(),
                "linked_file_or_page": str(row.get("linked_file_or_page", "")).strip(),
                "file_name": str(row.get("file_name", "")).strip(),
            })

    retrieval_df_local = pd.DataFrame(rows)

    if retrieval_df_local.empty:
        raise ValueError(
            f"No chunks were created for chunk_size={chunk_size_words}, "
            f"overlap={chunk_overlap_words}."
        )

    return retrieval_df_local


def make_rerank_text(row):
    """Create a compact text field for CrossEncoder reranking."""
    title = str(row.get("title", "")).strip()
    document_type = str(row.get("document_type", "")).strip()
    course_code = str(row.get("course_code", "")).strip()
    file_name = str(row.get("file_name", "")).strip()
    chunk_text = str(row.get("chunk_text", row.get("text", ""))).strip()

    return normalize_whitespace(
        f"Title: {title}\n"
        f"Document Type: {document_type}\n"
        f"Course Code: {course_code}\n"
        f"File: {file_name}\n"
        f"Content: {chunk_text}"
    )


def build_qrels_for_retrieval(retrieval_df_local):
    """
    Build qrels and rel_by_q for this chunking setup.

    This supports both:
    - qrels that already point directly to chunk IDs
    - qrels that point to original parent document IDs
    """
    if not qrels_available:
        return pd.DataFrame(columns=["query_id", "doc_id", "relevance"]), {}

    retrieval_doc_ids = set(retrieval_df_local["doc_id"].astype(str))

    # Direct chunk-id qrels, if any exist.
    direct_qrels = qrels_original_df[qrels_original_df["doc_id"].astype(str).isin(retrieval_doc_ids)].copy()

    # Parent-doc qrels expanded to chunk IDs.
    chunk_map = retrieval_df_local[["doc_id", "parent_doc_id"]].copy()
    chunk_map["doc_id"] = chunk_map["doc_id"].astype(str)
    chunk_map["parent_doc_id"] = chunk_map["parent_doc_id"].astype(str)
    chunk_map = chunk_map.rename(columns={"doc_id": "chunk_doc_id"})

    expanded_qrels = qrels_original_df.merge(
        chunk_map,
        left_on="doc_id",
        right_on="parent_doc_id",
        how="inner",
    )

    if not expanded_qrels.empty:
        expanded_qrels = (
            expanded_qrels[["query_id", "chunk_doc_id", "relevance"]]
            .rename(columns={"chunk_doc_id": "doc_id"})
            .copy()
        )

    qrels_parts = []
    if not direct_qrels.empty:
        qrels_parts.append(direct_qrels[["query_id", "doc_id", "relevance"]])
    if not expanded_qrels.empty:
        qrels_parts.append(expanded_qrels[["query_id", "doc_id", "relevance"]])

    if qrels_parts:
        qrels_df_local = (
            pd.concat(qrels_parts, ignore_index=True)
            .drop_duplicates(subset=["query_id", "doc_id"], keep="first")
            .reset_index(drop=True)
        )
    else:
        qrels_df_local = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])

    rel_by_q_local = {}
    for qid, group in qrels_df_local.groupby("query_id"):
        rel_by_q_local[str(qid)] = {
            str(r.doc_id): int(r.relevance)
            for r in group.itertuples(index=False)
        }

    return qrels_df_local, rel_by_q_local


def build_faiss_resources(retrieval_df_local):
    """Encode chunks and build a FAISS cosine-similarity index."""
    doc_ids_local = retrieval_df_local["doc_id"].astype(str).tolist()
    doc_texts_local = retrieval_df_local["text"].astype(str).tolist()

    safe_batch_size = int(globals().get("EMBED_BATCH_SIZE", 16))

    doc_embeddings_local = model.encode(
        doc_texts_local,
        batch_size=safe_batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    doc_embeddings_local = doc_embeddings_local.astype(np.float32)
    index_local = faiss.IndexFlatIP(doc_embeddings_local.shape[1])
    index_local.add(doc_embeddings_local)

    return doc_ids_local, doc_texts_local, doc_embeddings_local, index_local


def build_bm25_run(tokenized_corpus_local, doc_ids_local, k1, b, top_k):
    """Build a BM25 run dataframe for one BM25 parameter setup."""
    bm25_local = BM25Okapi(tokenized_corpus_local, k1=float(k1), b=float(b))
    safe_top_k = min(int(top_k), len(doc_ids_local))

    rows = []
    for _, qrow in queries_df.iterrows():
        qid = str(qrow["query_id"])
        query_tokens = tokenize_for_bm25(qrow["query_text"])
        scores = bm25_local.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:safe_top_k]

        for rank, idx in enumerate(top_indices, start=1):
            rows.append({
                "query_id": qid,
                "doc_id": str(doc_ids_local[idx]),
                "rank": int(rank),
                "score": float(scores[idx]),
            })

    return pd.DataFrame(rows)


def build_dense_run(index_local, doc_ids_local, top_k):
    """Build a dense FAISS run dataframe for one top_k."""
    safe_top_k = min(int(top_k), len(doc_ids_local), int(index_local.ntotal))

    rows = []
    for _, qrow in queries_df.iterrows():
        qid = str(qrow["query_id"])
        qtext = str(qrow["query_text"])

        query_vec = model.encode(
            [qtext],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)

        scores, indices = index_local.search(query_vec, safe_top_k)

        for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
            if int(idx) < 0:
                continue
            rows.append({
                "query_id": qid,
                "doc_id": str(doc_ids_local[int(idx)]),
                "rank": int(rank),
                "score": float(score),
            })

    return pd.DataFrame(rows)


def rrf_fuse_runs(bm25_df, dense_df, rrf_k=60, bm25_weight=1.0, dense_weight=1.0):
    """Fuse BM25 and dense runs with weighted reciprocal rank fusion."""
    fused_rows = []
    query_ids = sorted(
        set(bm25_df["query_id"].astype(str)).union(set(dense_df["query_id"].astype(str)))
    )

    for qid in query_ids:
        scores = {}

        bm25_q = bm25_df[bm25_df["query_id"].astype(str) == qid]
        for row in bm25_q.itertuples(index=False):
            doc_id = str(row.doc_id)
            rank = int(row.rank)
            scores[doc_id] = scores.get(doc_id, 0.0) + (float(bm25_weight) / (float(rrf_k) + rank))

        dense_q = dense_df[dense_df["query_id"].astype(str) == qid]
        for row in dense_q.itertuples(index=False):
            doc_id = str(row.doc_id)
            rank = int(row.rank)
            scores[doc_id] = scores.get(doc_id, 0.0) + (float(dense_weight) / (float(rrf_k) + rank))

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        for final_rank, (doc_id, score) in enumerate(ranked, start=1):
            fused_rows.append({
                "query_id": qid,
                "doc_id": doc_id,
                "rank": int(final_rank),
                "score": float(score),
            })

    return pd.DataFrame(fused_rows)


def average_precision_local(ranked_doc_ids, rel_map):
    """Binary AP. Relevance values above 0 count as relevant."""
    total_relevant = sum(1 for v in rel_map.values() if int(v) > 0)
    if total_relevant == 0:
        return 0.0

    hits = 0
    ap_sum = 0.0
    for i, doc_id in enumerate(ranked_doc_ids, start=1):
        if int(rel_map.get(str(doc_id), 0)) > 0:
            hits += 1
            ap_sum += hits / i

    return ap_sum / total_relevant


def ndcg_at_k_local(ranked_doc_ids, rel_map, k=30):
    """Graded nDCG@k using the relevance values from qrels."""
    dcg = 0.0
    for i, doc_id in enumerate(ranked_doc_ids[:k], start=1):
        gain = int(rel_map.get(str(doc_id), 0))
        dcg += gain / math.log2(i + 1)

    ideal_gains = sorted([int(v) for v in rel_map.values()], reverse=True)
    idcg = 0.0
    for i, gain in enumerate(ideal_gains[:k], start=1):
        idcg += gain / math.log2(i + 1)

    return (dcg / idcg) if idcg > 0 else 0.0


def evaluate_run_quiet(run_df, rel_by_q_local, k=30):
    """Return mean nDCG@k, MAP, and optional per-query diagnostics."""
    if not qrels_available:
        return np.nan, np.nan, pd.DataFrame()

    rows = []
    query_ids = queries_df["query_id"].astype(str).tolist()

    for qid in query_ids:
        q_run = run_df[run_df["query_id"].astype(str) == qid]
        ranked = q_run.sort_values("rank")["doc_id"].astype(str).tolist()
        rel_map = rel_by_q_local.get(str(qid), {})

        ap = average_precision_local(ranked, rel_map)
        ndcg = ndcg_at_k_local(ranked, rel_map, k=k)

        first_relevant_rank = None
        for rank_index, doc_id in enumerate(ranked, start=1):
            if int(rel_map.get(str(doc_id), 0)) > 0:
                first_relevant_rank = rank_index
                break

        relevant_retrieved = sum(1 for doc_id in ranked if int(rel_map.get(str(doc_id), 0)) > 0)
        total_relevant = sum(1 for v in rel_map.values() if int(v) > 0)

        rows.append({
            "query_id": qid,
            "ndcg": float(ndcg),
            "ap": float(ap),
            "first_relevant_rank": first_relevant_rank if first_relevant_rank is not None else "",
            "relevant_retrieved": int(relevant_retrieved),
            "total_relevant": int(total_relevant),
        })

    diagnostics_df = pd.DataFrame(rows)
    mean_ndcg = float(diagnostics_df["ndcg"].mean()) if not diagnostics_df.empty else 0.0
    mean_map = float(diagnostics_df["ap"].mean()) if not diagnostics_df.empty else 0.0

    return mean_ndcg, mean_map, diagnostics_df


def predict_reranker_scores(reranker_model, pairs, batch_size=16):
    """Handle minor sentence-transformers API differences across versions."""
    try:
        return reranker_model.predict(
            pairs,
            batch_size=int(batch_size),
            show_progress_bar=False,
        )
    except TypeError:
        return reranker_model.predict(
            pairs,
            batch_size=int(batch_size),
        )


def rerank_run_with_cross_encoder(run_df, queries_source_df, doc_text_lookup, reranker_model, rerank_top_n=50):
    """
    Rerank the top rerank_top_n documents per query with a CrossEncoder.
    Documents below rerank_top_n keep their original relative order after the reranked block.
    """
    if reranker_model is None or int(rerank_top_n) <= 0:
        return run_df.copy()

    query_lookup = {
        str(row.query_id): str(row.query_text)
        for row in queries_source_df[["query_id", "query_text"]].itertuples(index=False)
    }

    reranked_rows = []

    for qid in sorted(run_df["query_id"].astype(str).unique()):
        q_run = run_df[run_df["query_id"].astype(str) == qid].sort_values("rank").copy()
        query_text = query_lookup.get(str(qid), "")

        # Remove duplicate doc IDs while preserving original order.
        ordered_doc_ids = []
        seen = set()
        for doc_id in q_run["doc_id"].astype(str).tolist():
            if doc_id not in seen:
                ordered_doc_ids.append(doc_id)
                seen.add(doc_id)

        top_doc_ids = ordered_doc_ids[:int(rerank_top_n)]
        remaining_doc_ids = ordered_doc_ids[int(rerank_top_n):]

        pairs = [
            (query_text, doc_text_lookup.get(str(doc_id), ""))
            for doc_id in top_doc_ids
        ]

        if pairs:
            rerank_scores = predict_reranker_scores(
                reranker_model,
                pairs,
                batch_size=RERANKER_BATCH_SIZE,
            )
            rerank_scores = [float(s) for s in rerank_scores]
            reranked_top = sorted(
                zip(top_doc_ids, rerank_scores),
                key=lambda x: x[1],
                reverse=True,
            )
        else:
            reranked_top = []

        final_rank = 1
        for doc_id, score in reranked_top:
            reranked_rows.append({
                "query_id": qid,
                "doc_id": str(doc_id),
                "rank": int(final_rank),
                "score": float(score),
                "score_type": "cross_encoder",
            })
            final_rank += 1

        # Keep non-reranked tail after the reranked block.
        original_score_lookup = {
            str(row.doc_id): float(row.score)
            for row in q_run.itertuples(index=False)
            if hasattr(row, "score")
        }

        for doc_id in remaining_doc_ids:
            reranked_rows.append({
                "query_id": qid,
                "doc_id": str(doc_id),
                "rank": int(final_rank),
                "score": float(original_score_lookup.get(str(doc_id), 0.0)),
                "score_type": "original_rrf_tail",
            })
            final_rank += 1

    return pd.DataFrame(reranked_rows)


def combined_metric_score(ndcg_value, map_value):
    return (
        TUNING_COMBINED_NDCG_WEIGHT * float(ndcg_value)
        + TUNING_COMBINED_MAP_WEIGHT * float(map_value)
    )


def print_best_expanded_block(title, row):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    print(f"Method:               {row['method']}")
    print(f"Combined score:       {row['combined_score']:.4f}")
    print(f"nDCG@{TUNING_NDCG_K}:             {row['ndcg']:.4f}")
    print(f"MAP:                  {row['map']:.4f}")
    print(f"Chunk size words:     {int(row['chunk_size_words'])}")
    print(f"Chunk overlap words:  {int(row['chunk_overlap_words'])}")
    print(f"top_k:                {int(row['top_k'])}")
    print(f"BM25 k1:              {row['bm25_k1']}")
    print(f"BM25 b:               {row['bm25_b']}")
    print(f"RRF k:                {int(row['rrf_k'])}")
    print(f"BM25 weight:          {row['bm25_weight']}")
    print(f"Dense weight:         {row['dense_weight']}")
    print(f"Rerank top N:         {int(row['rerank_top_n'])}")
    print(f"Reranker model:       {row.get('reranker_model', '')}")


def build_artifacts_for_parameter_row(best_row, reranker_model=None):
    """Rebuild retrieval artifacts and the final run for a selected best parameter row."""
    chunk_size = int(best_row["chunk_size_words"])
    chunk_overlap = int(best_row["chunk_overlap_words"])
    top_k = int(best_row["top_k"])

    retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, chunk_size, chunk_overlap)
    qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)

    doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
    tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]

    bm25_df = build_bm25_run(
        tokenized_corpus_local,
        doc_ids_local,
        k1=float(best_row["bm25_k1"]),
        b=float(best_row["bm25_b"]),
        top_k=top_k,
    )

    dense_df = build_dense_run(index_local, doc_ids_local, top_k=top_k)

    rrf_df = rrf_fuse_runs(
        bm25_df,
        dense_df,
        rrf_k=int(best_row["rrf_k"]),
        bm25_weight=float(best_row["bm25_weight"]),
        dense_weight=float(best_row["dense_weight"]),
    )

    doc_text_lookup = {
        str(row.doc_id): make_rerank_text(row._asdict())
        for row in retrieval_df_local.itertuples(index=False)
    }

    if str(best_row["method"]) == "rrf_plus_rerank" and reranker_model is not None:
        final_run_df = rerank_run_with_cross_encoder(
            rrf_df,
            queries_df,
            doc_text_lookup,
            reranker_model,
            rerank_top_n=int(best_row["rerank_top_n"]),
        )
    else:
        final_run_df = rrf_df.copy()

    final_ndcg, final_map, diagnostics_df = evaluate_run_quiet(final_run_df, rel_by_q_local, k=TUNING_NDCG_K)

    return {
        "retrieval_df": retrieval_df_local,
        "qrels_df": qrels_df_local,
        "rel_by_q": rel_by_q_local,
        "doc_ids": doc_ids_local,
        "doc_texts": doc_texts_local,
        "doc_embeddings": doc_embeddings_local,
        "index": index_local,
        "tokenized_corpus": tokenized_corpus_local,
        "final_run_df": final_run_df,
        "diagnostics_df": diagnostics_df,
        "final_ndcg": final_ndcg,
        "final_map": final_map,
    }


# ============================================================
# Run tuning
# ============================================================
if not qrels_available:
    print("Skipping expanded tuning because search_review/eecs_qrels.csv was not found.")
    print("Add reviewed qrels first, then rerun this cell to optimize nDCG and MAP.")
else:
    start_time = time.time()

    # Load reranker once.
    reranker = None
    if ENABLE_RERANKING:
        try:
            from sentence_transformers import CrossEncoder
            print(f"Loading reranker: {RERANKER_MODEL_NAME}")
            reranker = CrossEncoder(RERANKER_MODEL_NAME)
            print("Reranker loaded.")
        except Exception as e:
            print("Warning: reranker could not be loaded. Continuing with Stage 1 RRF-only tuning.")
            print(f"Reranker error: {e}")
            reranker = None

    stage1_rows = []
    stage2_rows = []

    chunk_grid = [
        (chunk_size, chunk_overlap)
        for chunk_size, chunk_overlap in product(
            TUNING_CHUNK_SIZE_WORDS_VALUES,
            TUNING_CHUNK_OVERLAP_WORDS_VALUES,
        )
        if int(chunk_overlap) < int(chunk_size)
    ]

    retrieval_grid = list(product(
        TUNING_TOP_K_VALUES,
        TUNING_BM25_K1_VALUES,
        TUNING_BM25_B_VALUES,
        TUNING_RRF_K_VALUES,
        TUNING_BM25_WEIGHT_VALUES,
        TUNING_DENSE_WEIGHT_VALUES,
    ))

    print("\n" + "=" * 78)
    print("Expanded Stage 1 Search: Chunking + BM25 + Dense + RRF")
    print("=" * 78)
    print(f"Chunking combinations:       {len(chunk_grid)}")
    print(f"Retrieval combinations each: {len(retrieval_grid)}")
    print(f"Total Stage 1 combinations: {len(chunk_grid) * len(retrieval_grid)}")
    print(f"Metric target: nDCG@{TUNING_NDCG_K} and MAP")

    for chunk_number, (chunk_size, chunk_overlap) in enumerate(chunk_grid, start=1):
        print("\n" + "-" * 78)
        print(f"Chunk setup {chunk_number}/{len(chunk_grid)}: size={chunk_size}, overlap={chunk_overlap}")
        print("-" * 78)

        retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, chunk_size, chunk_overlap)
        qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)

        print(f"Retrieval chunks created: {len(retrieval_df_local)}")
        print(f"Expanded/local qrels rows: {len(qrels_df_local)}")

        doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
        tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]

        bm25_run_cache = {}
        dense_run_cache = {}

        for combo_number, (top_k, bm25_k1, bm25_b, rrf_k, bm25_weight, dense_weight) in enumerate(retrieval_grid, start=1):
            bm25_key = (float(bm25_k1), float(bm25_b), int(top_k))
            dense_key = int(top_k)

            if bm25_key not in bm25_run_cache:
                bm25_run_cache[bm25_key] = build_bm25_run(
                    tokenized_corpus_local,
                    doc_ids_local,
                    k1=bm25_k1,
                    b=bm25_b,
                    top_k=top_k,
                )

            if dense_key not in dense_run_cache:
                dense_run_cache[dense_key] = build_dense_run(
                    index_local,
                    doc_ids_local,
                    top_k=top_k,
                )

            candidate_rrf_df = rrf_fuse_runs(
                bm25_run_cache[bm25_key],
                dense_run_cache[dense_key],
                rrf_k=rrf_k,
                bm25_weight=bm25_weight,
                dense_weight=dense_weight,
            )

            ndcg_value, map_value, _ = evaluate_run_quiet(
                candidate_rrf_df,
                rel_by_q_local,
                k=TUNING_NDCG_K,
            )

            combined_score = combined_metric_score(ndcg_value, map_value)

            stage1_rows.append({
                "method": "rrf",
                "chunk_size_words": int(chunk_size),
                "chunk_overlap_words": int(chunk_overlap),
                "top_k": int(top_k),
                "bm25_k1": float(bm25_k1),
                "bm25_b": float(bm25_b),
                "rrf_k": int(rrf_k),
                "bm25_weight": float(bm25_weight),
                "dense_weight": float(dense_weight),
                "rerank_top_n": 0,
                "reranker_model": "",
                "ndcg_k": int(TUNING_NDCG_K),
                "ndcg": float(ndcg_value),
                "map": float(map_value),
                "combined_score": float(combined_score),
                "num_chunks": int(len(retrieval_df_local)),
                "num_qrels": int(len(qrels_df_local)),
            })

            if combo_number == 1 or combo_number % 500 == 0 or combo_number == len(retrieval_grid):
                current_chunk_best = max(
                    [r for r in stage1_rows if r["chunk_size_words"] == chunk_size and r["chunk_overlap_words"] == chunk_overlap],
                    key=lambda r: r["combined_score"],
                )
                print(
                    f"Checked {combo_number:>5}/{len(retrieval_grid)} for this chunk setup | "
                    f"chunk best combined={current_chunk_best['combined_score']:.4f}, "
                    f"nDCG@{TUNING_NDCG_K}={current_chunk_best['ndcg']:.4f}, "
                    f"MAP={current_chunk_best['map']:.4f}"
                )

    stage1_results_df = pd.DataFrame(stage1_rows).sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).reset_index(drop=True)
    stage1_results_df.to_csv(STAGE1_RESULTS_PATH, index=False)

    print("\n" + "=" * 78)
    print("Stage 1 complete.")
    print("=" * 78)
    print(f"Saved Stage 1 results to: {STAGE1_RESULTS_PATH}")
    print("Top 10 Stage 1 RRF-only configurations:")
    display(stage1_results_df.head(10))

    # ----------------------------
    # Stage 2 reranking pass
    # ----------------------------
    if reranker is not None and not stage1_results_df.empty:
        stage1_top_df = stage1_results_df.head(int(TUNING_STAGE1_KEEP_TOP_N)).copy()

        print("\n" + "=" * 78)
        print("Stage 2 Search: CrossEncoder Reranking of Best Stage 1 Configurations")
        print("=" * 78)
        print(f"Stage 1 configurations reranked: {len(stage1_top_df)}")
        print(f"Rerank-top-N values tested:       {TUNING_RERANK_TOP_N_VALUES}")
        print(f"Total Stage 2 combinations:      {len(stage1_top_df) * len(TUNING_RERANK_TOP_N_VALUES)}")

        # Group by chunking so embeddings/chunks only need to be rebuilt once per chunk setup.
        for (chunk_size, chunk_overlap), group_df in stage1_top_df.groupby(["chunk_size_words", "chunk_overlap_words"]):
            print("\n" + "-" * 78)
            print(f"Reranking chunk setup: size={chunk_size}, overlap={chunk_overlap}")
            print("-" * 78)

            retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, int(chunk_size), int(chunk_overlap))
            qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)
            doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
            tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]
            doc_text_lookup = {
                str(row.doc_id): make_rerank_text(row._asdict())
                for row in retrieval_df_local.itertuples(index=False)
            }

            bm25_run_cache = {}
            dense_run_cache = {}

            for row_number, candidate in enumerate(group_df.itertuples(index=False), start=1):
                top_k = int(candidate.top_k)
                bm25_key = (float(candidate.bm25_k1), float(candidate.bm25_b), int(top_k))
                dense_key = int(top_k)

                if bm25_key not in bm25_run_cache:
                    bm25_run_cache[bm25_key] = build_bm25_run(
                        tokenized_corpus_local,
                        doc_ids_local,
                        k1=float(candidate.bm25_k1),
                        b=float(candidate.bm25_b),
                        top_k=top_k,
                    )

                if dense_key not in dense_run_cache:
                    dense_run_cache[dense_key] = build_dense_run(
                        index_local,
                        doc_ids_local,
                        top_k=top_k,
                    )

                rrf_df = rrf_fuse_runs(
                    bm25_run_cache[bm25_key],
                    dense_run_cache[dense_key],
                    rrf_k=int(candidate.rrf_k),
                    bm25_weight=float(candidate.bm25_weight),
                    dense_weight=float(candidate.dense_weight),
                )

                for rerank_top_n in TUNING_RERANK_TOP_N_VALUES:
                    reranked_df = rerank_run_with_cross_encoder(
                        rrf_df,
                        queries_df,
                        doc_text_lookup,
                        reranker,
                        rerank_top_n=int(rerank_top_n),
                    )

                    ndcg_value, map_value, _ = evaluate_run_quiet(
                        reranked_df,
                        rel_by_q_local,
                        k=TUNING_NDCG_K,
                    )
                    combined_score = combined_metric_score(ndcg_value, map_value)

                    stage2_rows.append({
                        "method": "rrf_plus_rerank",
                        "chunk_size_words": int(chunk_size),
                        "chunk_overlap_words": int(chunk_overlap),
                        "top_k": int(top_k),
                        "bm25_k1": float(candidate.bm25_k1),
                        "bm25_b": float(candidate.bm25_b),
                        "rrf_k": int(candidate.rrf_k),
                        "bm25_weight": float(candidate.bm25_weight),
                        "dense_weight": float(candidate.dense_weight),
                        "rerank_top_n": int(rerank_top_n),
                        "reranker_model": RERANKER_MODEL_NAME,
                        "ndcg_k": int(TUNING_NDCG_K),
                        "ndcg": float(ndcg_value),
                        "map": float(map_value),
                        "combined_score": float(combined_score),
                        "num_chunks": int(len(retrieval_df_local)),
                        "num_qrels": int(len(qrels_df_local)),
                    })

                if row_number == 1 or row_number % 10 == 0 or row_number == len(group_df):
                    if stage2_rows:
                        current_rerank_best = max(stage2_rows, key=lambda r: r["combined_score"])
                        print(
                            f"Reranked {row_number:>3}/{len(group_df)} configs for this chunk setup | "
                            f"best reranked combined={current_rerank_best['combined_score']:.4f}, "
                            f"nDCG@{TUNING_NDCG_K}={current_rerank_best['ndcg']:.4f}, "
                            f"MAP={current_rerank_best['map']:.4f}"
                        )

    if stage2_rows:
        rerank_results_df = pd.DataFrame(stage2_rows).sort_values(
            ["combined_score", "ndcg", "map"],
            ascending=False,
        ).reset_index(drop=True)
        rerank_results_df.to_csv(RERANK_RESULTS_PATH, index=False)
    else:
        rerank_results_df = pd.DataFrame(columns=stage1_results_df.columns)

    # ----------------------------
    # Final selection across RRF-only and RRF+rerank results
    # ----------------------------
    all_results_df = pd.concat(
        [stage1_results_df, rerank_results_df],
        ignore_index=True,
    ).sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).reset_index(drop=True)

    all_results_df.to_csv(ALL_RESULTS_PATH, index=False)

    best_overall = all_results_df.iloc[0]
    best_ndcg = all_results_df.sort_values(
        ["ndcg", "map", "combined_score"],
        ascending=False,
    ).iloc[0]
    best_map = all_results_df.sort_values(
        ["map", "ndcg", "combined_score"],
        ascending=False,
    ).iloc[0]

    best_summary_df = pd.DataFrame([
        {"selection": "Best combined nDCG/MAP", **best_overall.to_dict()},
        {"selection": f"Best nDCG@{TUNING_NDCG_K}", **best_ndcg.to_dict()},
        {"selection": "Best MAP", **best_map.to_dict()},
    ])
    best_summary_df.to_csv(BEST_SUMMARY_PATH, index=False)

    # Rebuild and save the true best run plus its retrieval corpus.
    print("\nRebuilding best configuration for saved artifacts...")
    best_artifacts = build_artifacts_for_parameter_row(best_overall, reranker_model=reranker)

    run_best_expanded_tuned_df = best_artifacts["final_run_df"].copy()
    run_best_expanded_tuned_df.to_csv(BEST_RUN_PATH, index=False)

    best_retrieval_df = best_artifacts["retrieval_df"].copy()
    best_retrieval_df.to_csv(BEST_RETRIEVAL_CORPUS_PATH, index=False)

    if SAVE_BEST_PER_QUERY_DIAGNOSTICS:
        best_artifacts["diagnostics_df"].to_csv(BEST_PER_QUERY_DIAGNOSTICS_PATH, index=False)

    # Store best values as global variables for reuse in later notebook cells.
    BEST_TUNED_METHOD = str(best_overall["method"])
    BEST_TUNED_CHUNK_SIZE_WORDS = int(best_overall["chunk_size_words"])
    BEST_TUNED_CHUNK_OVERLAP_WORDS = int(best_overall["chunk_overlap_words"])
    BEST_TUNED_TOP_K = int(best_overall["top_k"])
    BEST_TUNED_BM25_K1 = float(best_overall["bm25_k1"])
    BEST_TUNED_BM25_B = float(best_overall["bm25_b"])
    BEST_TUNED_RRF_K = int(best_overall["rrf_k"])
    BEST_TUNED_BM25_WEIGHT = float(best_overall["bm25_weight"])
    BEST_TUNED_DENSE_WEIGHT = float(best_overall["dense_weight"])
    BEST_TUNED_RERANK_TOP_N = int(best_overall["rerank_top_n"])
    BEST_TUNED_RERANKER_MODEL = str(best_overall.get("reranker_model", ""))
    BEST_TUNED_NDCG = float(best_overall["ndcg"])
    BEST_TUNED_MAP = float(best_overall["map"])
    BEST_TUNED_COMBINED_SCORE = float(best_overall["combined_score"])

    # Make the best retrieval setup active for later cells.
    active_corpus_df = best_artifacts["retrieval_df"].copy()
    retrieval_df = active_corpus_df
    RETRIEVAL_IS_CHUNKED = "parent_doc_id" in active_corpus_df.columns

    doc_ids = best_artifacts["doc_ids"]
    doc_texts = best_artifacts["doc_texts"]
    doc_embeddings = best_artifacts["doc_embeddings"]
    index = best_artifacts["index"]
    tokenized_corpus = best_artifacts["tokenized_corpus"]

    # Build BM25 with the best BM25 settings for the final RAG loop.
    bm25 = BM25Okapi(
        tokenized_corpus,
        k1=BEST_TUNED_BM25_K1,
        b=BEST_TUNED_BM25_B,
    )

    # Refresh lookup dictionaries used by later RAG/context cells.
    doc_lookup = dict(zip(active_corpus_df["doc_id"], active_corpus_df["text"].astype(str)))
    metadata_lookup = active_corpus_df.set_index("doc_id").to_dict(orient="index")
    rel_by_q = best_artifacts["rel_by_q"]
    qrels_df = best_artifacts["qrels_df"].copy()

    # Keep compatibility with earlier naming.
    run_rrf_best_tuned_df = run_best_expanded_tuned_df.copy()
    run_rrf_df = run_best_expanded_tuned_df.copy()

    print_best_expanded_block("Best Overall Parameters by Combined nDCG/MAP", best_overall)
    print_best_expanded_block(f"Best Parameters by nDCG@{TUNING_NDCG_K}", best_ndcg)
    print_best_expanded_block("Best Parameters by MAP", best_map)

    elapsed_minutes = (time.time() - start_time) / 60

    print("\n" + "=" * 78)
    print("Saved Outputs")
    print("=" * 78)
    print(f"Stage 1 RRF results:          {STAGE1_RESULTS_PATH}")
    print(f"Rerank results:               {RERANK_RESULTS_PATH}")
    print(f"All tuning results:           {ALL_RESULTS_PATH}")
    print(f"Best summary:                 {BEST_SUMMARY_PATH}")
    print(f"Best tuned run:               {BEST_RUN_PATH}")
    print(f"Best tuned retrieval corpus:  {BEST_RETRIEVAL_CORPUS_PATH}")
    if SAVE_BEST_PER_QUERY_DIAGNOSTICS:
        print(f"Best per-query diagnostics:   {BEST_PER_QUERY_DIAGNOSTICS_PATH}")
    print(f"Elapsed time:                 {elapsed_minutes:.2f} minutes")

    print("\nTop 10 overall tuning configurations:")
    display(all_results_df.head(10))

    print("\nBest summary:")
    display(best_summary_df)

    if SAVE_BEST_PER_QUERY_DIAGNOSTICS:
        print("\nPer-query diagnostics for the selected best configuration:")
        display(best_artifacts["diagnostics_df"])


Loading reranker: BAAI/bge-reranker-base


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded.

Expanded Stage 1 Search: Chunking + BM25 + Dense + RRF
Chunking combinations:       12
Retrieval combinations each: 1728
Total Stage 1 combinations: 20736
Metric target: nDCG@30 and MAP

------------------------------------------------------------------------------
Chunk setup 1/12: size=300, overlap=50
------------------------------------------------------------------------------
Retrieval chunks created: 747
Expanded/local qrels rows: 223


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

...

,method,chunk_size_words,chunk_overlap_words,top_k,bm25_k1,bm25_b,rrf_k,bm25_weight,dense_weight,rerank_top_n,reranker_model,ndcg_k,ndcg,map,combined_score,num_chunks,num_qrels
0,rrf,700,100,100,2.2,1.0,150,0.50,2.0,0,,30,0.493974,0.351199,0.422587,582,182
1,rrf,700,100,100,2.2,1.0,150,0.25,2.5,0,,30,0.492954,0.351977,0.422466,582,182
2,rrf,700,100,100,1.8,1.0,150,0.25,3.0,0,,30,0.492857,0.352064,0.422461,582,182
3,rrf,700,100,100,2.2,1.0,150,0.25,3.0,0,,30,0.493016,0.351856,0.422436,582,182
4,rrf,700,100,100,1.8,1.0,150,0.25,2.5,0,,30,0.492784,0.351994,0.422389,582,182
5,rrf,700,100,100,2.2,1.0,90,0.25,2.0,0,,30,0.492849,0.351768,0.422309,582,182
6,rrf,700,100,100,2.2,1.0,120,0.25,2.5,0,,30,0.492856,0.351637,0.422246,582,182
7,rrf,700,100,100,2.2,1.0,150,0.25,2.0,0,,30,0.491583,0.352668,0.422125,582,182
8,rrf,700,100,100,1.8,1.0,120,0.25,2.5,0,,30,0.492648,0.351575,0.422112,582,182
9,rrf,700,100,100,1.8,0.9,150,0.50,2.0,0,,30,0.494245,0.349540,0.421892,582,182



Best summary:


,selection,method,chunk_size_words,chunk_overlap_words,top_k,bm25_k1,bm25_b,rrf_k,bm25_weight,dense_weight,rerank_top_n,reranker_model,ndcg_k,ndcg,map,combined_score,num_chunks,num_qrels
0,Best combined nDCG/MAP,rrf,700,100,100,2.2,1.0,150,0.5,2.0,0,,30,0.493974,0.351199,0.422587,582,182
1,Best nDCG@30,rrf,700,100,100,1.8,0.9,150,0.5,2.0,0,,30,0.494245,0.349540,0.421892,582,182
2,Best MAP,rrf,700,150,200,2.2,1.0,150,0.5,2.5,0,,30,0.487480,0.353582,0.420531,586,182



Per-query diagnostics for the selected best configuration:


,query_id,ndcg,ap,first_relevant_rank,relevant_retrieved,total_relevant
0,q01,0.765542,0.541306,1,12,12
1,q02,0.848461,0.660735,1,9,9
2,q03,0.636538,0.415000,2,4,4
3,q04,0.000000,0.007353,68,1,2
4,q05,0.200538,0.057881,10,5,6
5,q06,0.232541,0.082587,7,5,7
6,q07,0.405145,0.222578,7,12,12
7,q08,0.488156,0.319737,5,19,19
8,q09,0.596881,0.493023,2,22,22
9,q10,0.652715,0.482256,3,18,19


## Final Interactive Hybrid RAG Loop — Quality-Guarded Version

This final block keeps the retrieval and answer-generation steps separated. The user's original question is preserved for the LLM-facing prompt, while a retrieval-optimized version is used only for BM25 + FAISS + RRF. The retrieved chunks are compressed into query-relevant background facts, inserted into the Hugging Face prompt, and hidden from normal output. Only the final contextualized answer is displayed.

This version also includes answer-quality guards to reduce document-title copying, repeated-token failures, and irrelevant context leakage.

In [29]:
# # ============================================================
# # Final Interactive Hybrid RAG Loop
# # Keyword-Gated Retrieval + Conversation Continuation Version
# # ============================================================
# # Paste this whole block at the end of the notebook after:
# # - corpus_df / active_corpus_df have been created
# # - BM25 has been built as bm25
# # - FAISS has been built as index
# # - doc_ids has been created for the active retrieval dataframe
# # - dense_search(), tokenize_for_bm25(), and rrf_process() exist
# # - the dense embedding model variable is named model
# #
# # Behavior:
# # 1. User enters a query.
# # 2. If the query contains major EECS keywords, fresh BM25 + FAISS + RRF retrieval runs.
# # 3. If no major EECS keywords are found and previous retrieved context exists,
# #    retrieval is skipped and the previous context is reused.
# # 4. The LLM receives the original question plus hidden retrieved context.
# # 5. The user only sees the final contextualized answer.

# import os

# # ----------------------------
# # RAG settings
# # ----------------------------
# RAG_TOP_K = 10
# RAG_RETRIEVAL_POOL = 150
# RAG_RRF_K = 60

# # BM25-heavy weighting works well for course codes, forms, certificates, and degree sheets.
# RAG_BM25_WEIGHT = 2.6
# RAG_DENSE_WEIGHT = 0.25

# # RAG_MAX_CONTEXT_CHARS = 3500
# # RAG_MAX_SNIPPET_CHARS = 520
# # RAG_MAX_INPUT_TOKENS = 1536
# # RAG_MAX_NEW_TOKENS = 180
# # RAG_DEBUG = False
# RAG_MAX_INPUT_TOKENS = 2048
# RAG_MAX_CONTEXT_CHARS = 5500
# RAG_MAX_SNIPPET_CHARS = 600
# RAG_MAX_NEW_TOKENS = 220
# RAG_DEBUG = False


# # Recommended options:
# #   Qwen/Qwen2.5-3B-Instruct      -> stronger answers, heavier
# #   Qwen/Qwen2.5-1.5B-Instruct    -> lighter/faster
# #   Qwen/Qwen2.5-0.5B-Instruct    -> very small, weaker
# #   google/flan-t5-base           -> legacy seq2seq option
# RAG_MODEL_NAME = globals().get("RAG_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")

# # Optional Hugging Face token.
# # Either set HF_TOKEN in an earlier cell or set it as an environment variable.
# HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", None))
# if HF_TOKEN:
#     os.environ["HF_TOKEN"] = HF_TOKEN


# # ----------------------------
# # Answer model loading
# # ----------------------------
# def is_seq2seq_rag_model(model_name):
#     name = str(model_name).lower()
#     return "t5" in name or "flan" in name


# def load_rag_generation_model(model_name):
#     print(f"Loading Hugging Face RAG answer model: {model_name}")

#     tokenizer_kwargs = {"trust_remote_code": True}
#     if HF_TOKEN:
#         tokenizer_kwargs["token"] = HF_TOKEN

#     tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)

#     model_kwargs = {"trust_remote_code": True}
#     if HF_TOKEN:
#         model_kwargs["token"] = HF_TOKEN

#     if is_seq2seq_rag_model(model_name):
#         model_obj = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
#         model_kind = "seq2seq"
#     else:
#         # Use dtype instead of deprecated torch_dtype.
#         if torch.cuda.is_available():
#             model_kwargs.update({"device_map": "auto", "dtype": torch.float16})
#         else:
#             model_kwargs.update({"dtype": torch.float32})
#         model_obj = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
#         model_kind = "causal"

#     if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
#         tokenizer.pad_token = tokenizer.eos_token

#     if not torch.cuda.is_available():
#         model_obj = model_obj.to("cpu")

#     model_obj.eval()
#     print(f"RAG answer model loaded as: {model_kind}")
#     return tokenizer, model_obj, model_kind


# rag_tokenizer, rag_model, RAG_MODEL_KIND = load_rag_generation_model(RAG_MODEL_NAME)


# # ----------------------------
# # Query utilities
# # ----------------------------
# RAG_STOPWORDS = {
#     "a", "an", "and", "are", "as", "at", "be", "by", "can", "could", "do", "does",
#     "for", "from", "how", "i", "in", "is", "it", "me", "my", "need", "needed", "of",
#     "on", "or", "should", "the", "their", "there", "this", "to", "what", "when", "where",
#     "which", "who", "will", "with", "would", "you", "your", "about", "related", "information",
#     "student"
# }


# def rag_normalize_text(text):
#     return " ".join(str(text).replace("\x00", " ").split())


# def rag_terms(text):
#     words = re.findall(r"[A-Za-z][A-Za-z0-9]+", str(text).lower())
#     return [w for w in words if w not in RAG_STOPWORDS and len(w) > 2]


# def classify_rag_intent(original_query):
#     q = str(original_query).lower()

#     if any(t in q for t in ["recommend", "class", "classes", "course", "courses", "take", "fall", "spring", "summer", "senior"]):
#         return "course_recommendation"

#     if "certificate" in q or "certification" in q:
#         return "certificate_requirements"

#     if any(t in q for t in ["credit", "credits", "bachelor", "bachelors", "b.s.", "bs ", "undergraduate degree", "degree requirements"]):
#         return "degree_requirements"

#     if any(t in q for t in ["chair", "director", "advisor", "coordinator", "dean", "acting chair"]):
#         return "person_lookup"

#     if any(t in q for t in ["form", "worksheet", "application", "audit", "pos", "plan of study"]):
#         return "forms"

#     return "general"


# EECS_RETRIEVAL_EXPANSIONS_BY_INTENT = {
#     "course_recommendation": [
#         "course listing", "course offerings", "undergraduate", "senior", "4000", "3000", "fall",
#         "CAP", "CDA", "CEN", "CIS", "COP", "COT", "EEL", "EEE", "prerequisite", "credits"
#     ],
#     "certificate_requirements": [
#         "certificate", "requirements", "required courses", "credits", "worksheet", "program sheet",
#         "application", "big data analytics", "artificial intelligence", "cybersecurity", "data science"
#     ],
#     "degree_requirements": [
#         "degree requirements", "program sheet", "worksheet", "bachelor of science", "BS", "BSEE",
#         "undergraduate", "total credits", "required credits", "electrical engineering", "computer science"
#     ],
#     "person_lookup": [
#         "acting chair", "department chair", "chair", "director", "faculty",
#         "computer science engineering electrical engineering"
#     ],
#     "forms": [
#         "form", "forms", "worksheet", "application", "degree audit", "plan of study", "POS", "requirements"
#     ],
#     "general": [
#         "computer science", "electrical engineering", "department", "course", "program", "requirements"
#     ]
# }

# EECS_KEYWORD_EXPANSIONS = {
#     "ai": ["artificial intelligence", "machine learning"],
#     "artificial intelligence": ["AI", "machine learning"],
#     "ml": ["machine learning", "artificial intelligence"],
#     "big data": ["big data analytics", "data analytics", "data science"],
#     "cyber": ["cybersecurity", "cyber security", "security"],
#     "cybersecurity": ["cyber security", "security"],
#     "computer science": ["CS", "CSE", "COP", "COT", "CAP", "CEN", "computer science"],
#     "computer engineering": ["computer engineering", "CpE", "CDA", "EEE", "EEL"],
#     "electrical engineering": ["electrical engineering", "EE", "EEL", "EEE", "BSEE"],
#     "bachelors": ["bachelor", "bachelor of science", "BS", "undergraduate", "total credits"],
#     "bachelor": ["bachelor of science", "BS", "undergraduate", "total credits"],
#     "credits": ["credit hours", "total credits", "required credits"],
#     "fall": ["fall semester", "course offerings", "schedule"],
#     "senior": ["4000", "3000", "upper division", "undergraduate"],
# }


# def improve_query_for_hybrid_search(original_query, max_extra_terms=24):
#     """Expand the query for retrieval only. The original query is still used in the LLM prompt."""
#     original_query = str(original_query).strip()
#     lowered = original_query.lower()
#     intent = classify_rag_intent(original_query)
#     extra_terms = []

#     for term in EECS_RETRIEVAL_EXPANSIONS_BY_INTENT.get(intent, []):
#         if term.lower() not in lowered and term not in extra_terms:
#             extra_terms.append(term)

#     for trigger, expansions in EECS_KEYWORD_EXPANSIONS.items():
#         if trigger in lowered:
#             for term in expansions:
#                 if term.lower() not in lowered and term not in extra_terms:
#                     extra_terms.append(term)

#     extra_terms = extra_terms[:max_extra_terms]
#     return original_query if not extra_terms else original_query + " " + " ".join(extra_terms)


# # ----------------------------
# # Keyword gate for retrieval refresh
# # ----------------------------
# MAJOR_EECS_KEYWORDS = {
#     # Department/program names
#     "computer science", "cs", "cse", "electrical engineering", "ee", "computer engineering", "eecs", "engineering",

#     # Course prefixes
#     "cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee", "egm", "egn",

#     # Academic planning terms
#     "course", "courses", "class", "classes", "credit", "credits", "prerequisite", "prerequisites",
#     "requirement", "requirements", "degree", "bachelor", "bachelors", "undergraduate", "graduate",
#     "masters", "ms", "phd", "doctoral", "senior", "junior", "fall", "spring", "summer", "semester", "schedule",

#     # Forms/certificates
#     "form", "forms", "worksheet", "worksheets", "application", "plan of study", "pos", "degree audit", "certificate",

#     # Topics
#     "artificial intelligence", "ai", "machine learning", "ml", "data science", "big data", "big data analytics",
#     "cybersecurity", "cyber security", "security", "software engineering", "database", "databases", "networks",
#     "embedded", "robotics", "signals", "circuits", "systems", "programming", "algorithms",

#     # People/department roles
#     "chair", "acting chair", "director", "advisor", "coordinator", "faculty", "professor", "dean"
# }

# FOLLOWUP_CUES = {
#     "that", "those", "these", "them", "it", "they", "which", "what about", "how about", "why",
#     "explain", "continue", "more", "also", "compare", "rank", "best", "better", "easier", "harder",
#     "recommend", "summarize", "clarify", "second", "third", "first", "last"
# }


# def query_has_major_eecs_keyword(query_text):
#     """Return True if the query appears to introduce a new EECS retrieval topic."""
#     q = str(query_text).lower().strip()
#     if not q:
#         return False

#     for keyword in MAJOR_EECS_KEYWORDS:
#         if keyword in q:
#             return True

#     # Course-code style match: COP 3530, COT3100, EEL 3111, CAP4611, etc.
#     if re.search(r"\b(cap|cda|cen|cis|cop|cot|eel|eee|egm|egn)\s*\d{4}\b", q, flags=re.IGNORECASE):
#         return True

#     return False


# def query_looks_like_followup(query_text):
#     q = str(query_text).lower().strip()
#     if not q:
#         return False
#     return any(cue in q for cue in FOLLOWUP_CUES)


# def should_refresh_retrieval_for_query(query_text):
#     """If major EECS keywords are present, refresh retrieval; otherwise reuse last context."""
#     return query_has_major_eecs_keyword(query_text)


# # ----------------------------
# # Retrieval helpers
# # ----------------------------
# def bm25_search_custom(query_text, bm25_model, doc_ids, top_k=100):
#     query_tokens = tokenize_for_bm25(query_text)
#     scores = bm25_model.get_scores(query_tokens)
#     top_indices = np.argsort(scores)[::-1][:top_k]

#     return [
#         (doc_ids[idx], float(scores[idx]), rank)
#         for rank, idx in enumerate(top_indices, start=1)
#     ]


# def hybrid_search_custom_query(
#     retrieval_query_text,
#     top_k=RAG_TOP_K,
#     retrieval_pool=RAG_RETRIEVAL_POOL,
#     rrf_k=RAG_RRF_K,
#     bm25_weight=RAG_BM25_WEIGHT,
#     dense_weight=RAG_DENSE_WEIGHT
# ):
#     custom_qid = "custom_query"

#     dense_results = dense_search(
#         query_text=retrieval_query_text,
#         model=model,
#         index=index,
#         doc_ids=doc_ids,
#         top_k=retrieval_pool
#     )

#     dense_df = pd.DataFrame([
#         {"query_id": custom_qid, "doc_id": doc_id, "rank": rank, "score": score}
#         for doc_id, score, rank in dense_results
#     ])

#     bm25_results = bm25_search_custom(
#         query_text=retrieval_query_text,
#         bm25_model=bm25,
#         doc_ids=doc_ids,
#         top_k=retrieval_pool
#     )

#     bm25_df = pd.DataFrame([
#         {"query_id": custom_qid, "doc_id": doc_id, "rank": rank, "score": score}
#         for doc_id, score, rank in bm25_results
#     ])

#     fused_df = rrf_process(
#         bm25_df,
#         dense_df,
#         k=rrf_k,
#         bm25_weight=bm25_weight,
#         dense_weight=dense_weight
#     )

#     bm25_rank_lookup = bm25_df.set_index("doc_id")["rank"].to_dict()
#     bm25_score_lookup = bm25_df.set_index("doc_id")["score"].to_dict()
#     dense_rank_lookup = dense_df.set_index("doc_id")["rank"].to_dict()
#     dense_score_lookup = dense_df.set_index("doc_id")["score"].to_dict()

#     fused_df["bm25_rank"] = fused_df["doc_id"].map(bm25_rank_lookup)
#     fused_df["bm25_score"] = fused_df["doc_id"].map(bm25_score_lookup)
#     fused_df["dense_rank"] = fused_df["doc_id"].map(dense_rank_lookup)
#     fused_df["dense_score"] = fused_df["doc_id"].map(dense_score_lookup)

#     return fused_df.sort_values("rank").reset_index(drop=True)


# # ----------------------------
# # Context compression
# # ----------------------------
# def get_search_dataframe():
#     return globals().get("active_corpus_df", corpus_df)


# def get_document_record(doc_id):
#     search_df = get_search_dataframe()
#     matches = search_df.loc[search_df["doc_id"].astype(str) == str(doc_id)]

#     if matches.empty and "parent_doc_id" in search_df.columns:
#         matches = search_df.loc[search_df["parent_doc_id"].astype(str) == str(doc_id)]

#     if matches.empty:
#         return {"doc_id": str(doc_id), "text": "", "chunk_text": ""}

#     return matches.iloc[0].to_dict()


# def get_record_value(record, *names):
#     for name in names:
#         value = str(record.get(name, "")).strip()
#         if value and value.lower() not in {"nan", "none", "not found"}:
#             return value
#     return ""


# def parse_rag_document(doc_id):
#     record = get_document_record(doc_id)

#     content = (
#         get_record_value(record, "chunk_text")
#         or get_record_value(record, "content", "Content")
#         or get_record_value(record, "text")
#     )

#     return {
#         "doc_id": str(doc_id),
#         "parent_doc_id": get_record_value(record, "parent_doc_id") or str(doc_id),
#         "chunk_index": get_record_value(record, "chunk_index"),
#         "content": rag_normalize_text(content),
#         "title": get_record_value(record, "title", "Title"),
#         "document_type": get_record_value(record, "document_type", "Document_Type"),
#         "course_code": get_record_value(record, "course_code", "Course_Code"),
#         "prefix": get_record_value(record, "prefix", "Prefix"),
#         "number": get_record_value(record, "number", "Number"),
#         "level": get_record_value(record, "level", "Level"),
#         "file_name": get_record_value(record, "file_name", "File_Name"),
#         "source": get_record_value(record, "linked_file_or_page", "source_page", "Source_Page"),
#     }


# def split_into_candidate_sentences(text):
#     text = rag_normalize_text(text)
#     rough = re.split(r"(?<=[.!?])\s+|\s{2,}|\s+[•●]\s+", text)

#     sentences = []
#     for part in rough:
#         part = part.strip(" -\t\n\r")
#         if 25 <= len(part) <= 520:
#             sentences.append(part)
#         elif len(part) > 520:
#             for i in range(0, len(part), 360):
#                 chunk = part[i:i + 520].strip()
#                 if len(chunk) >= 25:
#                     sentences.append(chunk)

#     return sentences


# def intent_keywords(intent):
#     return {
#         "course_recommendation": [
#             "course", "prerequisite", "credits", "undergraduate", "senior", "fall", "3000", "4000",
#             "cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee"
#         ],
#         "certificate_requirements": [
#             "certificate", "requirements", "required", "credits", "course", "courses", "complete", "program",
#             "worksheet", "application"
#         ],
#         "degree_requirements": [
#             "degree", "requirements", "credits", "credit", "bachelor", "undergraduate", "program",
#             "curriculum", "total", "engineering"
#         ],
#         "person_lookup": ["chair", "acting", "director", "coordinator", "advisor", "faculty", "professor"],
#         "forms": ["form", "worksheet", "application", "audit", "plan", "study", "requirements"],
#         "general": []
#     }.get(intent, [])


# def score_text_for_query(text, original_query, retrieval_query, intent):
#     lowered = str(text).lower()
#     q_terms = set(rag_terms(original_query)) | set(rag_terms(retrieval_query))
#     score = 0.0

#     for term in q_terms:
#         if term in lowered:
#             score += 2.0

#     for term in intent_keywords(intent):
#         if str(term).lower() in lowered:
#             score += 1.5

#     # Penalize irrelevant healthcare carryover/noisy text.
#     for term in ["patient", "diagnosis", "procedure", "medication", "clinical", "heart disease"]:
#         if term in lowered:
#             score -= 8.0

#     # Penalize repeated-word PDF extraction failures.
#     words = rag_terms(lowered)
#     if len(words) >= 30:
#         most_common_count = max([words.count(w) for w in set(words)] or [0])
#         if most_common_count / max(len(words), 1) > 0.18:
#             score -= 6.0

#     return score


# def select_relevant_snippet(doc, original_query, retrieval_query, intent, max_chars=RAG_MAX_SNIPPET_CHARS):
#     candidate_text = doc.get("content", "")
#     sentences = split_into_candidate_sentences(candidate_text)

#     if not sentences:
#         return rag_normalize_text(candidate_text)[:max_chars]

#     scored = [(score_text_for_query(sent, original_query, retrieval_query, intent), sent) for sent in sentences]
#     scored.sort(key=lambda x: x[0], reverse=True)

#     selected = []
#     total_chars = 0

#     for score, sent in scored[:8]:
#         if score <= 0 and selected:
#             continue
#         if sent in selected:
#             continue
#         if total_chars + len(sent) > max_chars and selected:
#             continue

#         selected.append(sent)
#         total_chars += len(sent)

#         if total_chars >= max_chars:
#             break

#     if not selected:
#         selected = [sentences[0][:max_chars]]

#     return rag_normalize_text(" ".join(selected))[:max_chars].rstrip()


# def document_intent_boost(doc, original_query, retrieval_query, intent):
#     combined = " ".join([
#         doc.get("title", ""),
#         doc.get("document_type", ""),
#         doc.get("course_code", ""),
#         doc.get("file_name", ""),
#         doc.get("source", ""),
#         doc.get("content", "")[:1200]
#     ]).lower()

#     boost = 0.0

#     if intent == "course_recommendation":
#         if "course" in combined:
#             boost += 6
#         if any(prefix in combined for prefix in ["cap", "cda", "cen", "cis", "cop", "cot", "eel", "eee"]):
#             boost += 3
#         if any(level in combined for level in ["undergraduate", "3000", "4000", "senior"]):
#             boost += 3
#         if any(term in combined for term in ["phd", "doctoral"]):
#             boost -= 5

#     elif intent == "certificate_requirements":
#         if "certificate" in combined:
#             boost += 8
#         if any(term in combined for term in ["requirement", "required", "credits", "worksheet", "application"]):
#             boost += 4
#         if "big data" in original_query.lower() and "big data" in combined:
#             boost += 8

#     elif intent == "degree_requirements":
#         if any(term in combined for term in ["degree requirements", "program sheet", "worksheet", "curriculum"]):
#             boost += 6
#         if any(term in original_query.lower() for term in ["electrical", "ee"]):
#             if any(term in combined for term in ["electrical engineering", "bsee", "eel", "eee"]):
#                 boost += 8
#             if any(term in combined for term in ["computer science", "mscs", "phd"]):
#                 boost -= 4
#         if any(term in combined for term in ["bachelor", "undergraduate", "total credits", "credits"]):
#             boost += 5
#         if any(term in combined for term in ["phd", "doctoral"]):
#             boost -= 8

#     elif intent == "person_lookup":
#         if any(term in combined for term in ["chair", "acting chair", "faculty", "director"]):
#             boost += 6
#         if any(term in combined for term in ["syllabus", "homework", "assignment", "phd phd"]):
#             boost -= 6

#     elif intent == "forms":
#         if any(term in combined for term in ["form", "worksheet", "application", "audit", "plan of study"]):
#             boost += 6

#     return boost


# def get_fused_base_score(row):
#     for col in ["rrf_score", "score", "fusion_score"]:
#         if col in row and pd.notna(row.get(col)):
#             try:
#                 return float(row.get(col))
#             except Exception:
#                 pass
#     return 0.0


# def rerank_fused_results(fused_df, original_query, retrieval_query, top_k=RAG_TOP_K):
#     intent = classify_rag_intent(original_query)
#     reranked_rows = []

#     for _, row in fused_df.head(RAG_RETRIEVAL_POOL).iterrows():
#         doc_id = str(row["doc_id"])
#         doc = parse_rag_document(doc_id)

#         base_score = get_fused_base_score(row) * 100.0
#         text_score = score_text_for_query(
#             " ".join([
#                 doc.get("title", ""),
#                 doc.get("file_name", ""),
#                 doc.get("course_code", ""),
#                 doc.get("content", "")[:2500]
#             ]),
#             original_query,
#             retrieval_query,
#             intent
#         )
#         boost = document_intent_boost(doc, original_query, retrieval_query, intent)

#         new_row = row.to_dict()
#         new_row["rag_rerank_score"] = base_score + text_score + boost
#         new_row["rag_intent"] = intent
#         new_row["title"] = doc.get("title", "")
#         new_row["course_code"] = doc.get("course_code", "")
#         new_row["document_type"] = doc.get("document_type", "")
#         new_row["file_name"] = doc.get("file_name", "")
#         reranked_rows.append(new_row)

#     reranked_df = pd.DataFrame(reranked_rows)

#     if reranked_df.empty:
#         return fused_df.head(top_k).copy()

#     reranked_df = reranked_df.sort_values("rag_rerank_score", ascending=False).head(top_k).reset_index(drop=True)
#     reranked_df["rank"] = range(1, len(reranked_df) + 1)
#     return reranked_df


# def build_fact_line(rank, doc, snippet):
#     parts = []

#     if doc.get("course_code"):
#         parts.append(f"course {doc.get('course_code')}")
#     if doc.get("title"):
#         parts.append(f"titled {doc.get('title')}")
#     if doc.get("level"):
#         parts.append(f"level {doc.get('level')}")
#     if doc.get("document_type"):
#         parts.append(f"source type {doc.get('document_type')}")

#     descriptor = ", ".join(parts)

#     if descriptor:
#         return f"Fact {rank}: In a department source for {descriptor}, the relevant information says: {snippet}"

#     return f"Fact {rank}: {snippet}"


# def build_hidden_rag_context(reranked_df, original_query, retrieval_query, max_total_chars=RAG_MAX_CONTEXT_CHARS):
#     intent = classify_rag_intent(original_query)
#     fact_lines = []
#     review_rows = []
#     total = 0

#     for _, row in reranked_df.iterrows():
#         rank = int(row["rank"])
#         doc_id = str(row["doc_id"])
#         doc = parse_rag_document(doc_id)
#         snippet = select_relevant_snippet(doc, original_query, retrieval_query, intent)

#         if not snippet:
#             continue

#         fact_line = build_fact_line(rank, doc, snippet)

#         if total + len(fact_line) > max_total_chars and fact_lines:
#             break

#         fact_lines.append(fact_line)
#         total += len(fact_line)

#         review_rows.append({
#             "rank": rank,
#             "doc_id": doc_id,
#             "rag_rerank_score": row.get("rag_rerank_score", ""),
#             "rrf_score": row.get("rrf_score", row.get("score", "")),
#             "bm25_rank": row.get("bm25_rank", ""),
#             "dense_rank": row.get("dense_rank", ""),
#             "title": doc.get("title", ""),
#             "document_type": doc.get("document_type", ""),
#             "course_code": doc.get("course_code", ""),
#             "file_name": doc.get("file_name", ""),
#             "snippet": snippet
#         })

#     return "\n".join(fact_lines), pd.DataFrame(review_rows)


# # ----------------------------
# # Prompting and answer quality guard
# # ----------------------------
# def build_contextualized_prompt(original_user_query, hidden_context):
#     return f"""
# Answer the student question using only the department facts below.
# Use the facts as background context.
# Do not mention document numbers, document IDs, file names, retrieval, chunks, BM25, FAISS, or RRF.
# Give a direct advising-style answer.

# If the facts do not contain the answer, say:
# "The retrieved department documents do not provide enough information to answer that."

# Student question:
# {original_user_query}

# Department facts:
# {hidden_context}

# Answer:
# """.strip()


# def build_conversation_continuation_prompt(current_user_query, previous_user_query, hidden_context):
#     return f"""
# Continue the advising conversation using the same department facts from the previous retrieval.
# The current student message does not appear to introduce a new EECS retrieval topic, so treat it as a follow-up.

# Do not mention document numbers, document IDs, file names, retrieval, chunks, BM25, FAISS, RRF, or hidden context.
# Answer naturally and directly.

# If the previous department facts are not enough to answer the follow-up, say:
# "The previously retrieved department documents do not provide enough information to answer that."

# Previous student question:
# {previous_user_query}

# Current follow-up:
# {current_user_query}

# Previously retrieved department facts:
# {hidden_context}

# Answer:
# """.strip()


# def clean_generated_answer(answer):
#     answer = rag_normalize_text(answer)
#     answer = re.sub(r"^(Answer:|Final answer:|Final answer for the user:)\s*", "", answer, flags=re.IGNORECASE).strip()
#     return answer


# def repeated_word_problem(answer):
#     words = re.findall(r"[A-Za-z]+", str(answer).lower())
#     if len(words) < 20:
#         return False
#     counts = {w: words.count(w) for w in set(words)}
#     return max(counts.values()) / len(words) > 0.22


# def answer_looks_bad(answer):
#     if not answer or len(str(answer).strip()) < 12:
#         return True

#     lowered = str(answer).lower()
#     bad_markers = [
#         "doc id", "document 1", "document 2", "document 3", "document 4", "document 5", "document 6",
#         "rrf", "bm25", "faiss", "pdf_", "course_", "retrieved department context",
#         "file_name", "source_page", "chunk"
#     ]

#     if any(marker in lowered for marker in bad_markers):
#         return True

#     if repeated_word_problem(answer):
#         return True

#     if re.fullmatch(r"[A-Za-z0-9_\- .()]+\.pdf", str(answer).strip()):
#         return True

#     return False


# def extract_course_suggestions_from_review(top_docs_df, max_items=6):
#     suggestions = []
#     seen = set()

#     if top_docs_df is None or len(top_docs_df) == 0:
#         return suggestions

#     for _, row in top_docs_df.iterrows():
#         code = str(row.get("course_code", "")).strip()
#         title = str(row.get("title", "")).strip()
#         key = code or title

#         if not key or key.lower() in seen:
#             continue

#         seen.add(key.lower())

#         if code and title:
#             suggestions.append(f"{code} — {title}")
#         elif title:
#             suggestions.append(title)
#         elif code:
#             suggestions.append(code)

#         if len(suggestions) >= max_items:
#             break

#     return suggestions


# def extractive_fallback_answer(original_user_query, top_docs_df, continuation_mode=False):
#     intent = classify_rag_intent(original_user_query)

#     insufficient = (
#         "The previously retrieved department documents do not provide enough information to answer that."
#         if continuation_mode
#         else "The retrieved department documents do not provide enough information to answer that."
#     )

#     if top_docs_df is None or len(top_docs_df) == 0:
#         return insufficient

#     snippets = [str(s).strip() for s in top_docs_df.get("snippet", []) if str(s).strip()]
#     if not snippets:
#         return insufficient

#     if intent == "course_recommendation":
#         suggestions = extract_course_suggestions_from_review(top_docs_df)
#         if suggestions:
#             return (
#                 "Based on the retrieved department course information, good options to review include "
#                 + "; ".join(suggestions[:6])
#                 + ". The student should still check the current schedule, prerequisites, and advisor guidance before registering."
#             )

#     if intent == "person_lookup":
#         joined = " ".join(snippets[:4])
#         m = re.search(
#             r"(?:acting\s+chair|chair|director)\s*(?:is|:|-)?\s*([A-Z][A-Za-z.'-]+(?:\s+[A-Z][A-Za-z.'-]+){0,3})",
#             joined
#         )
#         if m:
#             return f"The retrieved department context indicates that {m.group(1)} is associated with that role."
#         return "The retrieved department documents do not provide enough information to identify that person or role."

#     strongest = []
#     for s in snippets[:3]:
#         s = re.sub(
#             r"\b(Document_Type|Title|Course_Code|File_Name|Source_Page|Linked_File_or_Page)\s*=\s*",
#             "",
#             s
#         )
#         strongest.append(s)

#     return "Based on the retrieved department context: " + " ".join(strongest)


# def get_model_device(model_obj):
#     try:
#         return next(model_obj.parameters()).device
#     except StopIteration:
#         return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# def build_chat_or_plain_prompt(prompt):
#     if RAG_MODEL_KIND != "causal":
#         return prompt

#     system_message = (
#         "You are a helpful academic advising assistant. "
#         "Use only the supplied department context. "
#         "Do not mention retrieval, chunks, filenames, document IDs, BM25, FAISS, or RRF. "
#         "If the context does not answer the question, say that the retrieved department documents do not provide enough information."
#     )

#     messages = [
#         {"role": "system", "content": system_message},
#         {"role": "user", "content": prompt},
#     ]

#     if hasattr(rag_tokenizer, "apply_chat_template"):
#         try:
#             return rag_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#         except Exception:
#             return system_message + "\n\n" + prompt

#     return system_message + "\n\n" + prompt


# def decode_generated_tokens(outputs, inputs):
#     if RAG_MODEL_KIND == "causal":
#         input_len = inputs["input_ids"].shape[-1]
#         generated = outputs[0][input_len:]
#         return rag_tokenizer.decode(generated, skip_special_tokens=True)

#     return rag_tokenizer.decode(outputs[0], skip_special_tokens=True)


# def run_rag_model_once(prompt, max_input_tokens, max_new_tokens):
#     formatted_prompt = build_chat_or_plain_prompt(prompt)

#     inputs = rag_tokenizer(
#         formatted_prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=max_input_tokens
#     )

#     device = get_model_device(rag_model)
#     inputs = {key: value.to(device) for key, value in inputs.items()}

#     generation_kwargs = {
#         "max_new_tokens": max_new_tokens,
#         "do_sample": False,
#         "no_repeat_ngram_size": 3,
#         "repetition_penalty": 1.15,
#         "pad_token_id": rag_tokenizer.pad_token_id,
#     }

#     if RAG_MODEL_KIND == "seq2seq":
#         generation_kwargs.update({
#             "num_beams": 4,
#             "early_stopping": True,
#             "repetition_penalty": 1.25,
#         })
#     else:
#         if rag_tokenizer.eos_token_id is not None:
#             generation_kwargs["eos_token_id"] = rag_tokenizer.eos_token_id

#     with torch.no_grad():
#         outputs = rag_model.generate(**inputs, **generation_kwargs)

#     return clean_generated_answer(decode_generated_tokens(outputs, inputs))


# def generate_rag_answer(
#     original_user_query,
#     hidden_context,
#     top_docs_df=None,
#     max_input_tokens=RAG_MAX_INPUT_TOKENS,
#     max_new_tokens=RAG_MAX_NEW_TOKENS,
#     continuation_mode=False,
#     previous_user_query=None
# ):
#     if continuation_mode:
#         prompt = build_conversation_continuation_prompt(
#             current_user_query=original_user_query,
#             previous_user_query=previous_user_query or "Previous question not available.",
#             hidden_context=hidden_context
#         )
#     else:
#         prompt = build_contextualized_prompt(
#             original_user_query=original_user_query,
#             hidden_context=hidden_context
#         )

#     answer = run_rag_model_once(prompt, max_input_tokens=max_input_tokens, max_new_tokens=max_new_tokens)

#     if answer_looks_bad(answer):
#         shorter_context = "\n".join(str(hidden_context).splitlines()[:5])

#         if continuation_mode:
#             retry_prompt = build_conversation_continuation_prompt(
#                 current_user_query=original_user_query,
#                 previous_user_query=previous_user_query or "Previous question not available.",
#                 hidden_context=shorter_context
#             )
#         else:
#             retry_prompt = build_contextualized_prompt(
#                 original_user_query=original_user_query,
#                 hidden_context=shorter_context
#             )

#         retry_answer = run_rag_model_once(retry_prompt, max_input_tokens=1024, max_new_tokens=140)

#         if not answer_looks_bad(retry_answer):
#             return retry_answer

#         return extractive_fallback_answer(original_user_query, top_docs_df, continuation_mode=continuation_mode)

#     return answer


# # ----------------------------
# # Main RAG function and loop
# # ----------------------------
# LAST_RAG_RESULT = None


# def ask_department_rag(
#     original_user_query,
#     top_k=RAG_TOP_K,
#     retrieval_pool=RAG_RETRIEVAL_POOL,
#     return_details=False,
#     debug=RAG_DEBUG
# ):
#     global LAST_RAG_RESULT

#     original_user_query = str(original_user_query).strip()

#     if not original_user_query:
#         print("Please enter a non-empty query.")
#         return None

#     previous_result = LAST_RAG_RESULT if isinstance(LAST_RAG_RESULT, dict) else None

#     has_previous_context = (
#         previous_result is not None
#         and str(previous_result.get("hidden_context", "")).strip()
#         and previous_result.get("top_docs") is not None
#     )

#     should_refresh = should_refresh_retrieval_for_query(original_user_query)

#     # First usable question always retrieves, even if no keyword is detected.
#     if not has_previous_context:
#         should_refresh = True

#     if should_refresh:
#         retrieval_query = improve_query_for_hybrid_search(original_user_query)

#         fused_df = hybrid_search_custom_query(
#             retrieval_query_text=retrieval_query,
#             top_k=top_k,
#             retrieval_pool=retrieval_pool
#         )

#         reranked_df = rerank_fused_results(
#             fused_df=fused_df,
#             original_query=original_user_query,
#             retrieval_query=retrieval_query,
#             top_k=top_k
#         )

#         hidden_context, top_docs_df = build_hidden_rag_context(
#             reranked_df=reranked_df,
#             original_query=original_user_query,
#             retrieval_query=retrieval_query
#         )

#         continuation_mode = False
#         previous_user_query = None

#     else:
#         # No major EECS keyword detected. Reuse the previous context.
#         retrieval_query = previous_result.get("retrieval_query", "")
#         hidden_context = previous_result.get("hidden_context", "")
#         top_docs_df = previous_result.get("top_docs")
#         fused_df = previous_result.get("fused_results")
#         reranked_df = previous_result.get("reranked_results")

#         continuation_mode = True
#         previous_user_query = previous_result.get("original_query", "")

#     if not str(hidden_context).strip():
#         answer = (
#             "The previously retrieved department documents do not provide enough information to answer that."
#             if continuation_mode
#             else "The retrieved department documents do not provide enough information to answer that."
#         )
#     else:
#         answer = generate_rag_answer(
#             original_user_query=original_user_query,
#             hidden_context=hidden_context,
#             top_docs_df=top_docs_df,
#             continuation_mode=continuation_mode,
#             previous_user_query=previous_user_query
#         )

#     LAST_RAG_RESULT = {
#         "original_query": original_user_query,
#         "retrieval_query": retrieval_query,
#         "answer": answer,
#         "top_docs": top_docs_df,
#         "hidden_context": hidden_context,
#         "fused_results": fused_df,
#         "reranked_results": reranked_df,
#         "intent": classify_rag_intent(original_user_query),
#         "retrieval_refreshed": should_refresh,
#         "continuation_mode": continuation_mode,
#         "previous_user_query": previous_user_query,
#         "has_major_eecs_keyword": query_has_major_eecs_keyword(original_user_query),
#         "looks_like_followup": query_looks_like_followup(original_user_query)
#     }

#     print("\n" + "=" * 90)
#     print("Contextualized RAG Answer")
#     print("=" * 90)
#     print(answer)

#     if debug:
#         print("\nDebug mode is on. The retrieved chunks are shown below for inspection only.")
#         print("Original query:", original_user_query)
#         print("Retrieval query:", retrieval_query)
#         print("Retrieval refreshed:", should_refresh)
#         print("Continuation mode:", continuation_mode)
#         print("Major EECS keyword detected:", query_has_major_eecs_keyword(original_user_query))
#         print("Looks like follow-up:", query_looks_like_followup(original_user_query))

#         try:
#             display(top_docs_df)
#         except Exception:
#             print(top_docs_df)

#     if return_details:
#         return LAST_RAG_RESULT

#     return answer


# def rag_query_loop():
#     print("Interactive CS/EE Department RAG Assistant")
#     print("Enter a custom question.")
#     print("Fresh hybrid search runs when the query contains major EECS keywords.")
#     print("Follow-up questions without major EECS keywords reuse the previous retrieved context.")
#     print("Only the final contextualized answer is displayed.")
#     print("Type 'exit', 'quit', or 'q' to stop.")

#     while True:
#         user_query = input("\nEnter your CS/EE department query: ").strip()

#         if user_query.lower() in {"exit", "quit", "q"}:
#             print("Stopping interactive RAG loop.")
#             break

#         if not user_query:
#             print("Please enter a question or type 'exit' to stop.")
#             continue

#         _ = ask_department_rag(user_query, return_details=False, debug=False)


# # Start the interactive loop.
# rag_query_loop()



## Stage 3 Focused Refinement Around the Best Retrieval Configuration

This optional cell takes the strongest result from the expanded tuning loop and performs a narrower second pass around it. It tests nearby chunk sizes, overlap values, top-k values, RRF settings, BM25 settings, dense/BM25 weights, and a smaller/larger reranking window.

Run this after the expanded tuning cell if the best result appears concentrated around one parameter area.


In [30]:
# ============================================================
# Stage 3 Focused Refinement Around the Current Best Result
# ============================================================
# Purpose:
# - Start from the best configuration found in expanded tuning.
# - Search more tightly around the winning chunking/retrieval settings.
# - Test values just beyond the previous winning boundaries.
# - Rerank only the strongest focused RRF candidates.
# - Print/save the best focused configuration, including chunk size and overlap.
#
# Run this after the expanded tuning cell. It expects the helper functions from that cell
# to already exist in the notebook.

from itertools import product
from pathlib import Path
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# ----------------------------
# Stage 3 switches
# ----------------------------
STAGE3_ENABLE_RERANKING = True
STAGE3_RERANKER_MODEL_NAME = globals().get("RERANKER_MODEL_NAME", "BAAI/bge-reranker-base")
STAGE3_RERANKER_BATCH_SIZE = int(globals().get("RERANKER_BATCH_SIZE", 16))

# How many focused RRF configurations should be passed into the reranker stage.
# Increase this for a deeper but slower search.
STAGE3_KEEP_TOP_RRF_FOR_RERANK = 40

# How many chunk setups from Stage 3A should be used in Stage 3B.
STAGE3_KEEP_TOP_CHUNKS = 5

# Metric settings
STAGE3_NDCG_K = int(globals().get("TUNING_NDCG_K", 30))
STAGE3_COMBINED_NDCG_WEIGHT = 0.5
STAGE3_COMBINED_MAP_WEIGHT = 0.5

# Output paths
STAGE3_OUTPUT_DIR = Path("outputs")
STAGE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE3_CHUNK_SCAN_PATH = STAGE3_OUTPUT_DIR / "stage3_chunk_scan_results.csv"
STAGE3_RRF_RESULTS_PATH = STAGE3_OUTPUT_DIR / "stage3_focused_rrf_results.csv"
STAGE3_RERANK_RESULTS_PATH = STAGE3_OUTPUT_DIR / "stage3_focused_rerank_results.csv"
STAGE3_ALL_RESULTS_PATH = STAGE3_OUTPUT_DIR / "stage3_focused_all_results.csv"
STAGE3_BEST_SUMMARY_PATH = STAGE3_OUTPUT_DIR / "stage3_best_focused_summary.csv"
STAGE3_BEST_RUN_PATH = STAGE3_OUTPUT_DIR / "stage3_run_best_focused_tuned.csv"
STAGE3_BEST_CORPUS_PATH = STAGE3_OUTPUT_DIR / "stage3_best_focused_retrieval_corpus.csv"
STAGE3_BEST_DIAGNOSTICS_PATH = STAGE3_OUTPUT_DIR / "stage3_best_focused_per_query_diagnostics.csv"


# ============================================================
# Select baseline from earlier tuning output
# ============================================================
if "all_results_df" in globals() and isinstance(all_results_df, pd.DataFrame) and not all_results_df.empty:
    STAGE3_BASELINE_ROW = all_results_df.sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).iloc[0].copy()
elif "stage1_results_df" in globals() and isinstance(stage1_results_df, pd.DataFrame) and not stage1_results_df.empty:
    STAGE3_BASELINE_ROW = stage1_results_df.sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).iloc[0].copy()
else:
    # Fallback based on the best row reported by the latest run.
    STAGE3_BASELINE_ROW = pd.Series({
        "method": "rrf",
        "chunk_size_words": 700,
        "chunk_overlap_words": 100,
        "top_k": 100,
        "bm25_k1": 2.2,
        "bm25_b": 1.0,
        "rrf_k": 150,
        "bm25_weight": 0.50,
        "dense_weight": 2.0,
        "rerank_top_n": 0,
        "reranker_model": "",
        "ndcg": 0.493974,
        "map": 0.351199,
        "combined_score": 0.422587,
    })

print("\n" + "=" * 78)
print("Stage 3 Focused Refinement Baseline")
print("=" * 78)
print(STAGE3_BASELINE_ROW[[
    "method",
    "chunk_size_words",
    "chunk_overlap_words",
    "top_k",
    "bm25_k1",
    "bm25_b",
    "rrf_k",
    "bm25_weight",
    "dense_weight",
    "rerank_top_n",
    "ndcg",
    "map",
    "combined_score",
]])


# ============================================================
# Focused search grids
# ============================================================
# The current best is concentrated around chunk_size=700, overlap=100,
# top_k=100, b=1.0, rrf_k=150, and dense-heavy fusion.
# These grids test values around that area and just beyond prior boundaries.

STAGE3_CHUNK_SIZE_WORDS_VALUES = sorted(set([
    550, 600, 650, 700, 750, 800, 850,
    int(STAGE3_BASELINE_ROW["chunk_size_words"]),
]))

STAGE3_CHUNK_OVERLAP_WORDS_VALUES = sorted(set([
    75, 100, 125, 150, 175,
    int(STAGE3_BASELINE_ROW["chunk_overlap_words"]),
]))

# Stage 3A uses a small retrieval set to identify the strongest nearby chunking layouts.
STAGE3_CHUNK_SCAN_TOP_K_VALUES = sorted(set([100, 150, int(STAGE3_BASELINE_ROW["top_k"])]))
STAGE3_CHUNK_SCAN_RRF_K_VALUES = sorted(set([120, 150, 180, int(STAGE3_BASELINE_ROW["rrf_k"])]))
STAGE3_CHUNK_SCAN_BM25_WEIGHT_VALUES = sorted(set([0.25, 0.50, float(STAGE3_BASELINE_ROW["bm25_weight"])]))
STAGE3_CHUNK_SCAN_DENSE_WEIGHT_VALUES = sorted(set([2.0, 2.5, 3.0, float(STAGE3_BASELINE_ROW["dense_weight"])]))

# Stage 3B uses a more detailed retrieval grid, but only on the best chunking layouts.
STAGE3_TOP_K_VALUES = sorted(set([75, 100, 125, 150, 200, 250, 300, int(STAGE3_BASELINE_ROW["top_k"])]))
STAGE3_BM25_K1_VALUES = sorted(set([1.8, 2.0, 2.2, 2.4, 2.6, float(STAGE3_BASELINE_ROW["bm25_k1"])]))
STAGE3_BM25_B_VALUES = sorted(set([0.90, 0.95, 1.0, float(STAGE3_BASELINE_ROW["bm25_b"])]))
STAGE3_RRF_K_VALUES = sorted(set([90, 120, 150, 180, 210, 240, 300, int(STAGE3_BASELINE_ROW["rrf_k"])]))
STAGE3_BM25_WEIGHT_VALUES = sorted(set([0.0, 0.10, 0.25, 0.40, 0.50, 0.75, float(STAGE3_BASELINE_ROW["bm25_weight"])]))
STAGE3_DENSE_WEIGHT_VALUES = sorted(set([1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.5, 4.0, float(STAGE3_BASELINE_ROW["dense_weight"])]))

# Reranking did not beat RRF in the previous top 10, so this tests smaller and larger rerank windows.
STAGE3_RERANK_TOP_N_VALUES = [10, 20, 30, 50, 75, 100, 150]


def stage3_combined_score(ndcg_value, map_value):
    return (
        STAGE3_COMBINED_NDCG_WEIGHT * float(ndcg_value)
        + STAGE3_COMBINED_MAP_WEIGHT * float(map_value)
    )


def stage3_print_best_block(title, row):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    print(f"Method:               {row['method']}")
    print(f"Combined score:       {row['combined_score']:.6f}")
    print(f"nDCG@{STAGE3_NDCG_K}:             {row['ndcg']:.6f}")
    print(f"MAP:                  {row['map']:.6f}")
    print(f"Chunk size words:     {int(row['chunk_size_words'])}")
    print(f"Chunk overlap words:  {int(row['chunk_overlap_words'])}")
    print(f"top_k:                {int(row['top_k'])}")
    print(f"BM25 k1:              {row['bm25_k1']}")
    print(f"BM25 b:               {row['bm25_b']}")
    print(f"RRF k:                {int(row['rrf_k'])}")
    print(f"BM25 weight:          {row['bm25_weight']}")
    print(f"Dense weight:         {row['dense_weight']}")
    print(f"Rerank top N:         {int(row['rerank_top_n'])}")
    print(f"Reranker model:       {row.get('reranker_model', '')}")
    print(f"Number of chunks:     {int(row['num_chunks'])}")
    print(f"Number of qrels:      {int(row['num_qrels'])}")


def stage3_build_rrf_for_params(
    retrieval_df_local,
    doc_ids_local,
    tokenized_corpus_local,
    index_local,
    bm25_run_cache,
    dense_run_cache,
    top_k,
    bm25_k1,
    bm25_b,
    rrf_k,
    bm25_weight,
    dense_weight,
):
    """Build or reuse BM25/dense runs, then return one weighted RRF run."""
    bm25_key = (float(bm25_k1), float(bm25_b), int(top_k))
    dense_key = int(top_k)

    if bm25_key not in bm25_run_cache:
        bm25_run_cache[bm25_key] = build_bm25_run(
            tokenized_corpus_local,
            doc_ids_local,
            k1=float(bm25_k1),
            b=float(bm25_b),
            top_k=int(top_k),
        )

    if dense_key not in dense_run_cache:
        dense_run_cache[dense_key] = build_dense_run(
            index_local,
            doc_ids_local,
            top_k=int(top_k),
        )

    return rrf_fuse_runs(
        bm25_run_cache[bm25_key],
        dense_run_cache[dense_key],
        rrf_k=int(rrf_k),
        bm25_weight=float(bm25_weight),
        dense_weight=float(dense_weight),
    )


# ============================================================
# Run Stage 3 focused refinement
# ============================================================
if not globals().get("qrels_available", False):
    print("Skipping Stage 3 because qrels are not available.")
else:
    stage3_start_time = time.time()

    # ----------------------------
    # Load or reuse reranker
    # ----------------------------
    stage3_reranker = None
    if STAGE3_ENABLE_RERANKING:
        if "reranker" in globals() and reranker is not None:
            stage3_reranker = reranker
            print(f"Reusing already-loaded reranker: {STAGE3_RERANKER_MODEL_NAME}")
        else:
            try:
                from sentence_transformers import CrossEncoder
                print(f"Loading Stage 3 reranker: {STAGE3_RERANKER_MODEL_NAME}")
                stage3_reranker = CrossEncoder(STAGE3_RERANKER_MODEL_NAME)
                print("Stage 3 reranker loaded.")
            except Exception as e:
                print("Warning: Stage 3 reranker could not be loaded. Continuing with RRF-only focused search.")
                print(f"Reranker error: {e}")
                stage3_reranker = None

    # ----------------------------
    # Stage 3A: scan nearby chunking layouts
    # ----------------------------
    stage3_chunk_rows = []
    chunk_grid = [
        (chunk_size, chunk_overlap)
        for chunk_size, chunk_overlap in product(
            STAGE3_CHUNK_SIZE_WORDS_VALUES,
            STAGE3_CHUNK_OVERLAP_WORDS_VALUES,
        )
        if int(chunk_overlap) < int(chunk_size)
    ]

    chunk_scan_grid = list(product(
        STAGE3_CHUNK_SCAN_TOP_K_VALUES,
        [float(STAGE3_BASELINE_ROW["bm25_k1"])],
        [float(STAGE3_BASELINE_ROW["bm25_b"])],
        STAGE3_CHUNK_SCAN_RRF_K_VALUES,
        STAGE3_CHUNK_SCAN_BM25_WEIGHT_VALUES,
        STAGE3_CHUNK_SCAN_DENSE_WEIGHT_VALUES,
    ))

    print("\n" + "=" * 78)
    print("Stage 3A: Focused Chunking Scan")
    print("=" * 78)
    print(f"Chunking combinations:       {len(chunk_grid)}")
    print(f"Retrieval combinations each: {len(chunk_scan_grid)}")
    print(f"Total Stage 3A combinations: {len(chunk_grid) * len(chunk_scan_grid)}")

    for chunk_number, (chunk_size, chunk_overlap) in enumerate(chunk_grid, start=1):
        print(f"\nStage 3A chunk {chunk_number}/{len(chunk_grid)}: size={chunk_size}, overlap={chunk_overlap}")

        retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, int(chunk_size), int(chunk_overlap))
        qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)
        doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
        tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]

        bm25_run_cache = {}
        dense_run_cache = {}

        best_for_chunk = None

        for top_k, bm25_k1, bm25_b, rrf_k, bm25_weight, dense_weight in chunk_scan_grid:
            rrf_df = stage3_build_rrf_for_params(
                retrieval_df_local,
                doc_ids_local,
                tokenized_corpus_local,
                index_local,
                bm25_run_cache,
                dense_run_cache,
                top_k,
                bm25_k1,
                bm25_b,
                rrf_k,
                bm25_weight,
                dense_weight,
            )

            ndcg_value, map_value, _ = evaluate_run_quiet(rrf_df, rel_by_q_local, k=STAGE3_NDCG_K)
            combined = stage3_combined_score(ndcg_value, map_value)

            row = {
                "method": "rrf",
                "chunk_size_words": int(chunk_size),
                "chunk_overlap_words": int(chunk_overlap),
                "top_k": int(top_k),
                "bm25_k1": float(bm25_k1),
                "bm25_b": float(bm25_b),
                "rrf_k": int(rrf_k),
                "bm25_weight": float(bm25_weight),
                "dense_weight": float(dense_weight),
                "rerank_top_n": 0,
                "reranker_model": "",
                "ndcg_k": int(STAGE3_NDCG_K),
                "ndcg": float(ndcg_value),
                "map": float(map_value),
                "combined_score": float(combined),
                "num_chunks": int(len(retrieval_df_local)),
                "num_qrels": int(len(qrels_df_local)),
            }
            stage3_chunk_rows.append(row)

            if best_for_chunk is None or row["combined_score"] > best_for_chunk["combined_score"]:
                best_for_chunk = row

        print(
            f"Best for this chunk setup: combined={best_for_chunk['combined_score']:.6f}, "
            f"nDCG@{STAGE3_NDCG_K}={best_for_chunk['ndcg']:.6f}, MAP={best_for_chunk['map']:.6f}"
        )

    stage3_chunk_scan_df = pd.DataFrame(stage3_chunk_rows).sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).reset_index(drop=True)
    stage3_chunk_scan_df.to_csv(STAGE3_CHUNK_SCAN_PATH, index=False)

    print("\nTop Stage 3A chunking candidates:")
    display(stage3_chunk_scan_df.head(10))

    best_chunk_pairs_df = (
        stage3_chunk_scan_df
        .drop_duplicates(subset=["chunk_size_words", "chunk_overlap_words"])
        .head(int(STAGE3_KEEP_TOP_CHUNKS))
        [["chunk_size_words", "chunk_overlap_words"]]
        .copy()
    )

    selected_chunk_pairs = [
        (int(row.chunk_size_words), int(row.chunk_overlap_words))
        for row in best_chunk_pairs_df.itertuples(index=False)
    ]

    print("\nSelected chunk setups for Stage 3B:")
    print(selected_chunk_pairs)

    # ----------------------------
    # Stage 3B: focused retrieval grid on selected chunks
    # ----------------------------
    stage3_rrf_rows = []
    retrieval_grid = list(product(
        STAGE3_TOP_K_VALUES,
        STAGE3_BM25_K1_VALUES,
        STAGE3_BM25_B_VALUES,
        STAGE3_RRF_K_VALUES,
        STAGE3_BM25_WEIGHT_VALUES,
        STAGE3_DENSE_WEIGHT_VALUES,
    ))

    print("\n" + "=" * 78)
    print("Stage 3B: Focused Retrieval Search on Best Chunking Layouts")
    print("=" * 78)
    print(f"Selected chunk setups:        {len(selected_chunk_pairs)}")
    print(f"Retrieval combinations each:  {len(retrieval_grid)}")
    print(f"Total Stage 3B combinations:  {len(selected_chunk_pairs) * len(retrieval_grid)}")

    for chunk_number, (chunk_size, chunk_overlap) in enumerate(selected_chunk_pairs, start=1):
        print("\n" + "-" * 78)
        print(f"Stage 3B chunk setup {chunk_number}/{len(selected_chunk_pairs)}: size={chunk_size}, overlap={chunk_overlap}")
        print("-" * 78)

        retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, int(chunk_size), int(chunk_overlap))
        qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)
        doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
        tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]

        bm25_run_cache = {}
        dense_run_cache = {}

        for combo_number, (top_k, bm25_k1, bm25_b, rrf_k, bm25_weight, dense_weight) in enumerate(retrieval_grid, start=1):
            rrf_df = stage3_build_rrf_for_params(
                retrieval_df_local,
                doc_ids_local,
                tokenized_corpus_local,
                index_local,
                bm25_run_cache,
                dense_run_cache,
                top_k,
                bm25_k1,
                bm25_b,
                rrf_k,
                bm25_weight,
                dense_weight,
            )

            ndcg_value, map_value, _ = evaluate_run_quiet(rrf_df, rel_by_q_local, k=STAGE3_NDCG_K)
            combined = stage3_combined_score(ndcg_value, map_value)

            stage3_rrf_rows.append({
                "method": "rrf",
                "chunk_size_words": int(chunk_size),
                "chunk_overlap_words": int(chunk_overlap),
                "top_k": int(top_k),
                "bm25_k1": float(bm25_k1),
                "bm25_b": float(bm25_b),
                "rrf_k": int(rrf_k),
                "bm25_weight": float(bm25_weight),
                "dense_weight": float(dense_weight),
                "rerank_top_n": 0,
                "reranker_model": "",
                "ndcg_k": int(STAGE3_NDCG_K),
                "ndcg": float(ndcg_value),
                "map": float(map_value),
                "combined_score": float(combined),
                "num_chunks": int(len(retrieval_df_local)),
                "num_qrels": int(len(qrels_df_local)),
            })

            if combo_number == 1 or combo_number % 1000 == 0 or combo_number == len(retrieval_grid):
                current_best = max(stage3_rrf_rows, key=lambda r: r["combined_score"])
                print(
                    f"Checked {combo_number:>5}/{len(retrieval_grid)} for this chunk setup | "
                    f"overall Stage 3B best combined={current_best['combined_score']:.6f}, "
                    f"nDCG@{STAGE3_NDCG_K}={current_best['ndcg']:.6f}, MAP={current_best['map']:.6f}"
                )

    stage3_rrf_results_df = pd.DataFrame(stage3_rrf_rows).sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).reset_index(drop=True)
    stage3_rrf_results_df.to_csv(STAGE3_RRF_RESULTS_PATH, index=False)

    print("\nTop Stage 3B focused RRF configurations:")
    display(stage3_rrf_results_df.head(10))

    # ----------------------------
    # Stage 3C: rerank strongest focused candidates
    # ----------------------------
    stage3_rerank_rows = []

    if stage3_reranker is not None and not stage3_rrf_results_df.empty:
        stage3_top_for_rerank_df = stage3_rrf_results_df.head(int(STAGE3_KEEP_TOP_RRF_FOR_RERANK)).copy()

        print("\n" + "=" * 78)
        print("Stage 3C: Rerank Strongest Focused RRF Configurations")
        print("=" * 78)
        print(f"RRF configurations reranked: {len(stage3_top_for_rerank_df)}")
        print(f"Rerank top N values:         {STAGE3_RERANK_TOP_N_VALUES}")

        for (chunk_size, chunk_overlap), group_df in stage3_top_for_rerank_df.groupby(["chunk_size_words", "chunk_overlap_words"]):
            print("\n" + "-" * 78)
            print(f"Stage 3C reranking chunk setup: size={chunk_size}, overlap={chunk_overlap}")
            print("-" * 78)

            retrieval_df_local = build_tuned_retrieval_corpus(corpus_df, int(chunk_size), int(chunk_overlap))
            qrels_df_local, rel_by_q_local = build_qrels_for_retrieval(retrieval_df_local)
            doc_ids_local, doc_texts_local, doc_embeddings_local, index_local = build_faiss_resources(retrieval_df_local)
            tokenized_corpus_local = [tokenize_for_bm25(text) for text in retrieval_df_local["text"].astype(str).tolist()]
            doc_text_lookup = {
                str(row.doc_id): make_rerank_text(row._asdict())
                for row in retrieval_df_local.itertuples(index=False)
            }

            bm25_run_cache = {}
            dense_run_cache = {}

            for row_number, candidate in enumerate(group_df.itertuples(index=False), start=1):
                rrf_df = stage3_build_rrf_for_params(
                    retrieval_df_local,
                    doc_ids_local,
                    tokenized_corpus_local,
                    index_local,
                    bm25_run_cache,
                    dense_run_cache,
                    int(candidate.top_k),
                    float(candidate.bm25_k1),
                    float(candidate.bm25_b),
                    int(candidate.rrf_k),
                    float(candidate.bm25_weight),
                    float(candidate.dense_weight),
                )

                for rerank_top_n in STAGE3_RERANK_TOP_N_VALUES:
                    reranked_df = rerank_run_with_cross_encoder(
                        rrf_df,
                        queries_df,
                        doc_text_lookup,
                        stage3_reranker,
                        rerank_top_n=int(rerank_top_n),
                    )

                    ndcg_value, map_value, _ = evaluate_run_quiet(reranked_df, rel_by_q_local, k=STAGE3_NDCG_K)
                    combined = stage3_combined_score(ndcg_value, map_value)

                    stage3_rerank_rows.append({
                        "method": "rrf_plus_rerank",
                        "chunk_size_words": int(chunk_size),
                        "chunk_overlap_words": int(chunk_overlap),
                        "top_k": int(candidate.top_k),
                        "bm25_k1": float(candidate.bm25_k1),
                        "bm25_b": float(candidate.bm25_b),
                        "rrf_k": int(candidate.rrf_k),
                        "bm25_weight": float(candidate.bm25_weight),
                        "dense_weight": float(candidate.dense_weight),
                        "rerank_top_n": int(rerank_top_n),
                        "reranker_model": STAGE3_RERANKER_MODEL_NAME,
                        "ndcg_k": int(STAGE3_NDCG_K),
                        "ndcg": float(ndcg_value),
                        "map": float(map_value),
                        "combined_score": float(combined),
                        "num_chunks": int(len(retrieval_df_local)),
                        "num_qrels": int(len(qrels_df_local)),
                    })

                if row_number == 1 or row_number % 10 == 0 or row_number == len(group_df):
                    if stage3_rerank_rows:
                        best_rerank_so_far = max(stage3_rerank_rows, key=lambda r: r["combined_score"])
                        print(
                            f"Reranked {row_number:>3}/{len(group_df)} configs for this chunk setup | "
                            f"best reranked combined={best_rerank_so_far['combined_score']:.6f}, "
                            f"nDCG@{STAGE3_NDCG_K}={best_rerank_so_far['ndcg']:.6f}, MAP={best_rerank_so_far['map']:.6f}"
                        )

    if stage3_rerank_rows:
        stage3_rerank_results_df = pd.DataFrame(stage3_rerank_rows).sort_values(
            ["combined_score", "ndcg", "map"],
            ascending=False,
        ).reset_index(drop=True)
        stage3_rerank_results_df.to_csv(STAGE3_RERANK_RESULTS_PATH, index=False)
    else:
        stage3_rerank_results_df = pd.DataFrame(columns=stage3_rrf_results_df.columns)

    # ----------------------------
    # Final Stage 3 selection
    # ----------------------------
    previous_results_df = all_results_df.copy() if "all_results_df" in globals() and isinstance(all_results_df, pd.DataFrame) else pd.DataFrame()

    stage3_all_results_df = pd.concat(
        [previous_results_df, stage3_rrf_results_df, stage3_rerank_results_df],
        ignore_index=True,
    ).drop_duplicates(
        subset=[
            "method",
            "chunk_size_words",
            "chunk_overlap_words",
            "top_k",
            "bm25_k1",
            "bm25_b",
            "rrf_k",
            "bm25_weight",
            "dense_weight",
            "rerank_top_n",
            "reranker_model",
        ],
        keep="first",
    ).sort_values(
        ["combined_score", "ndcg", "map"],
        ascending=False,
    ).reset_index(drop=True)

    stage3_all_results_df.to_csv(STAGE3_ALL_RESULTS_PATH, index=False)

    stage3_best_overall = stage3_all_results_df.iloc[0]
    stage3_best_ndcg = stage3_all_results_df.sort_values(
        ["ndcg", "map", "combined_score"],
        ascending=False,
    ).iloc[0]
    stage3_best_map = stage3_all_results_df.sort_values(
        ["map", "ndcg", "combined_score"],
        ascending=False,
    ).iloc[0]

    stage3_best_summary_df = pd.DataFrame([
        {"selection": "Stage 3 best combined nDCG/MAP", **stage3_best_overall.to_dict()},
        {"selection": f"Stage 3 best nDCG@{STAGE3_NDCG_K}", **stage3_best_ndcg.to_dict()},
        {"selection": "Stage 3 best MAP", **stage3_best_map.to_dict()},
    ])
    stage3_best_summary_df.to_csv(STAGE3_BEST_SUMMARY_PATH, index=False)

    # Rebuild/save the winning configuration and make it active for later RAG cells.
    print("\nRebuilding Stage 3 best configuration for saved artifacts and active retrieval...")
    stage3_best_artifacts = build_artifacts_for_parameter_row(stage3_best_overall, reranker_model=stage3_reranker)

    stage3_run_best_focused_tuned_df = stage3_best_artifacts["final_run_df"].copy()
    stage3_run_best_focused_tuned_df.to_csv(STAGE3_BEST_RUN_PATH, index=False)

    stage3_best_retrieval_df = stage3_best_artifacts["retrieval_df"].copy()
    stage3_best_retrieval_df.to_csv(STAGE3_BEST_CORPUS_PATH, index=False)

    stage3_best_artifacts["diagnostics_df"].to_csv(STAGE3_BEST_DIAGNOSTICS_PATH, index=False)

    # Store best values as globals.
    STAGE3_BEST_TUNED_METHOD = str(stage3_best_overall["method"])
    STAGE3_BEST_TUNED_CHUNK_SIZE_WORDS = int(stage3_best_overall["chunk_size_words"])
    STAGE3_BEST_TUNED_CHUNK_OVERLAP_WORDS = int(stage3_best_overall["chunk_overlap_words"])
    STAGE3_BEST_TUNED_TOP_K = int(stage3_best_overall["top_k"])
    STAGE3_BEST_TUNED_BM25_K1 = float(stage3_best_overall["bm25_k1"])
    STAGE3_BEST_TUNED_BM25_B = float(stage3_best_overall["bm25_b"])
    STAGE3_BEST_TUNED_RRF_K = int(stage3_best_overall["rrf_k"])
    STAGE3_BEST_TUNED_BM25_WEIGHT = float(stage3_best_overall["bm25_weight"])
    STAGE3_BEST_TUNED_DENSE_WEIGHT = float(stage3_best_overall["dense_weight"])
    STAGE3_BEST_TUNED_RERANK_TOP_N = int(stage3_best_overall["rerank_top_n"])
    STAGE3_BEST_TUNED_RERANKER_MODEL = str(stage3_best_overall.get("reranker_model", ""))
    STAGE3_BEST_TUNED_NDCG = float(stage3_best_overall["ndcg"])
    STAGE3_BEST_TUNED_MAP = float(stage3_best_overall["map"])
    STAGE3_BEST_TUNED_COMBINED_SCORE = float(stage3_best_overall["combined_score"])

    # Also update the generic BEST_TUNED_* globals so later cells use the Stage 3 winner.
    BEST_TUNED_METHOD = STAGE3_BEST_TUNED_METHOD
    BEST_TUNED_CHUNK_SIZE_WORDS = STAGE3_BEST_TUNED_CHUNK_SIZE_WORDS
    BEST_TUNED_CHUNK_OVERLAP_WORDS = STAGE3_BEST_TUNED_CHUNK_OVERLAP_WORDS
    BEST_TUNED_TOP_K = STAGE3_BEST_TUNED_TOP_K
    BEST_TUNED_BM25_K1 = STAGE3_BEST_TUNED_BM25_K1
    BEST_TUNED_BM25_B = STAGE3_BEST_TUNED_BM25_B
    BEST_TUNED_RRF_K = STAGE3_BEST_TUNED_RRF_K
    BEST_TUNED_BM25_WEIGHT = STAGE3_BEST_TUNED_BM25_WEIGHT
    BEST_TUNED_DENSE_WEIGHT = STAGE3_BEST_TUNED_DENSE_WEIGHT
    BEST_TUNED_RERANK_TOP_N = STAGE3_BEST_TUNED_RERANK_TOP_N
    BEST_TUNED_RERANKER_MODEL = STAGE3_BEST_TUNED_RERANKER_MODEL
    BEST_TUNED_NDCG = STAGE3_BEST_TUNED_NDCG
    BEST_TUNED_MAP = STAGE3_BEST_TUNED_MAP
    BEST_TUNED_COMBINED_SCORE = STAGE3_BEST_TUNED_COMBINED_SCORE

    # Make the Stage 3 best retrieval setup active for later cells.
    active_corpus_df = stage3_best_artifacts["retrieval_df"].copy()
    retrieval_df = active_corpus_df
    RETRIEVAL_IS_CHUNKED = "parent_doc_id" in active_corpus_df.columns

    doc_ids = stage3_best_artifacts["doc_ids"]
    doc_texts = stage3_best_artifacts["doc_texts"]
    doc_embeddings = stage3_best_artifacts["doc_embeddings"]
    index = stage3_best_artifacts["index"]
    tokenized_corpus = stage3_best_artifacts["tokenized_corpus"]

    bm25 = BM25Okapi(
        tokenized_corpus,
        k1=BEST_TUNED_BM25_K1,
        b=BEST_TUNED_BM25_B,
    )

    doc_lookup = dict(zip(active_corpus_df["doc_id"], active_corpus_df["text"].astype(str)))
    metadata_lookup = active_corpus_df.set_index("doc_id").to_dict(orient="index")
    rel_by_q = stage3_best_artifacts["rel_by_q"]
    qrels_df = stage3_best_artifacts["qrels_df"].copy()

    run_rrf_best_tuned_df = stage3_run_best_focused_tuned_df.copy()
    run_rrf_df = stage3_run_best_focused_tuned_df.copy()

    stage3_print_best_block("Stage 3 Best Overall Parameters by Combined nDCG/MAP", stage3_best_overall)
    stage3_print_best_block(f"Stage 3 Best Parameters by nDCG@{STAGE3_NDCG_K}", stage3_best_ndcg)
    stage3_print_best_block("Stage 3 Best Parameters by MAP", stage3_best_map)

    elapsed_minutes = (time.time() - stage3_start_time) / 60

    print("\n" + "=" * 78)
    print("Stage 3 Saved Outputs")
    print("=" * 78)
    print(f"Chunk scan results:        {STAGE3_CHUNK_SCAN_PATH}")
    print(f"Focused RRF results:       {STAGE3_RRF_RESULTS_PATH}")
    print(f"Focused rerank results:    {STAGE3_RERANK_RESULTS_PATH}")
    print(f"All Stage 3 results:       {STAGE3_ALL_RESULTS_PATH}")
    print(f"Best Stage 3 summary:      {STAGE3_BEST_SUMMARY_PATH}")
    print(f"Best Stage 3 run:          {STAGE3_BEST_RUN_PATH}")
    print(f"Best Stage 3 corpus:       {STAGE3_BEST_CORPUS_PATH}")
    print(f"Best Stage 3 diagnostics:  {STAGE3_BEST_DIAGNOSTICS_PATH}")
    print(f"Elapsed time:              {elapsed_minutes:.2f} minutes")

    print("\nTop 15 Stage 3 overall configurations:")
    display(stage3_all_results_df.head(15))

    print("\nStage 3 best summary:")
    display(stage3_best_summary_df)

    print("\nPer-query diagnostics for the Stage 3 selected best configuration:")
    display(stage3_best_artifacts["diagnostics_df"])




Stage 3 Focused Refinement Baseline
method                      rrf
chunk_size_words            700
chunk_overlap_words         100
top_k                       100
bm25_k1                     2.2
bm25_b                      1.0
rrf_k                       150
bm25_weight                 0.5
dense_weight                2.0
rerank_top_n                  0
ndcg                   0.493974
map                    0.351199
combined_score         0.422587
Name: 0, dtype: object
Reusing already-loaded reranker: BAAI/bge-reranker-base

Stage 3A: Focused Chunking Scan
Chunking combinations:       35
Retrieval combinations each: 36
Total Stage 3A combinations: 1260

Stage 3A chunk 1/35: size=550, overlap=75


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Best for this chunk setup: combined=0.414968, nDCG@30=0.484733, MAP=0.345203

Stage 3A chunk 2/35: size=550, overlap=100


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Best for this chunk setup: combined=0.415287, nDCG@30=0.485841, MAP=0.344734

Stage 3A chunk 3/35: size=550, overlap=125


...

,method,chunk_size_words,chunk_overlap_words,top_k,bm25_k1,bm25_b,rrf_k,bm25_weight,dense_weight,rerank_top_n,reranker_model,ndcg_k,ndcg,map,combined_score,num_chunks,num_qrels
0,rrf,750,75,75,2.4,1.0,300,0.40,3.00,0,,30,0.501216,0.354042,0.427629,577,181
1,rrf,750,75,75,2.4,1.0,300,0.25,2.50,0,,30,0.500997,0.354228,0.427613,577,181
2,rrf,750,75,75,2.4,1.0,300,0.40,4.00,0,,30,0.500997,0.354228,0.427613,577,181
3,rrf,750,75,75,2.4,1.0,300,0.40,3.50,0,,30,0.500508,0.354331,0.427420,577,181
4,rrf,750,75,75,2.4,1.0,300,0.25,2.25,0,,30,0.500413,0.354294,0.427353,577,181
5,rrf,750,75,75,2.4,1.0,240,0.25,2.00,0,,30,0.499959,0.354122,0.427041,577,181
6,rrf,750,75,75,2.4,1.0,240,0.50,4.00,0,,30,0.499959,0.354122,0.427041,577,181
7,rrf,750,75,75,2.4,1.0,300,0.25,3.00,0,,30,0.499708,0.354354,0.427031,577,181
8,rrf,750,75,75,2.2,1.0,300,0.25,2.75,0,,30,0.499889,0.353981,0.426935,577,181
9,rrf,750,75,75,2.4,1.0,240,0.40,3.50,0,,30,0.500142,0.353694,0.426918,577,181



Stage 3 best summary:


,selection,method,chunk_size_words,chunk_overlap_words,top_k,bm25_k1,bm25_b,rrf_k,bm25_weight,dense_weight,rerank_top_n,reranker_model,ndcg_k,ndcg,map,combined_score,num_chunks,num_qrels
0,Stage 3 best combined nDCG/MAP,rrf,750,75,75,2.4,1.00,300,0.40,3.0,0,,30,0.501216,0.354042,0.427629,577,181
1,Stage 3 best nDCG@30,rrf,750,75,75,2.4,1.00,300,0.40,3.0,0,,30,0.501216,0.354042,0.427629,577,181
2,Stage 3 best MAP,rrf,750,75,75,1.8,0.95,300,0.25,4.0,0,,30,0.496365,0.356379,0.426372,577,181



Per-query diagnostics for the Stage 3 selected best configuration:


,query_id,ndcg,ap,first_relevant_rank,relevant_retrieved,total_relevant
0,q01,0.786376,0.529085,1,11,12
1,q02,0.851216,0.673036,1,9,9
2,q03,0.634691,0.412037,2,4,4
3,q04,0.000000,0.008475,59,1,2
4,q05,0.209266,0.063873,7,5,6
5,q06,0.241940,0.092792,7,5,7
6,q07,0.413336,0.239664,7,12,12
7,q08,0.527159,0.348842,5,19,19
8,q09,0.592748,0.480013,2,20,22
9,q10,0.658106,0.494733,3,18,19
